# Ardent Mills — Automated ETL Pipeline (Oracle)
**Source :** `Ardent_Mills_Data.xlsx` — 6 sheets  
**Target :** Oracle database — `ARD_OPS_*` tables  
**Mode   :** Incremental / Upsert — reruns update existing business keys and insert only missing rows using Oracle `MERGE`

## Cell 1 — Install Dependencies (run once)

In [112]:
# Uncomment and run once to install
# !pip install pandas openpyxl oracledb

## Cell 2 — Imports And Helpers

In [113]:
from pathlib import Path
import pandas as pd

import hashlib
import json
import logging
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import oracledb


if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

LOGGER = logging.getLogger("ardent_etl")
if not LOGGER.handlers:
    LOGGER.setLevel(logging.INFO)
    LOGGER.addHandler(logging.StreamHandler(sys.stdout))

SOURCE_SHEETS = ["Pack", "Mill", "Sales", "WorkOrder", "BinCleaning", "Fills"]
CREATED_BY ='Maruthi_R_M'
UNKNOWN_TEXT = "UNKNOWN"


@dataclass(frozen=True)
class TableSpec:
    source_sheet: str
    business_key: list[str]
    count_rule: str
    grain_description: str
    loader_key: list[str]
    physical_table_name: str | None = None


TABLE_SPECS: dict[str, TableSpec] = {
    "ARD_OPS_Site": TableSpec("BinCleaning", ["site_id"], "unique_business_key", "One row per site_id.", ["site_id"]),
    "ARD_OPS_ItemClass": TableSpec("Sales", ["item_class_id"], "unique_business_key", "One row per item class.", ["item_class_desc"]),
    "ARD_OPS_Product": TableSpec("Sales/Fills", ["product_id"], "unique_business_key", "One row per product.", ["product_id"]),
    "ARD_OPS_Customer": TableSpec("Sales", ["customer_id"], "unique_business_key", "One row per customer.", ["customer_nm"]),
    "ARD_OPS_ShipToAccount": TableSpec("Fills", ["ship_to_id"], "unique_business_key", "One row per ship-to account.", ["ship_to_id"]),
    "ARD_OPS_Carrier": TableSpec("Fills", ["carrier_id"], "unique_business_key", "One row per carrier.", ["carrier_code"]),
    "ARD_OPS_ProductionMix": TableSpec("Mill", ["production_mix_id"], "unique_business_key", "One row per production mix code.", ["production_mix_code"]),
    "ARD_OPS_MaintenanceType": TableSpec("WorkOrder", ["maintenance_type_id"], "unique_business_key", "One row per maintenance type.", ["maintenance_type"]),
    "ARD_OPS_CleaningType": TableSpec("BinCleaning", ["cleaning_type_id"], "unique_business_key", "One row per cleaning type.", ["cleaning_type_id"]),
    "ARD_OPS_PackLine": TableSpec("Pack", ["line_id"], "unique_business_key", "One row per site and line.", ["site_id", "line_name"]),
    "ARD_OPS_Bin": TableSpec("BinCleaning", ["bin_id"], "unique_business_key", "One row per bin.", ["bin_id"]),
    "ARD_OPS_PackRun": TableSpec("Pack", ["pack_run_id"], "same_as_source", "One row per source pack record.", ["pack_run_id"]),
    "ARD_OPS_MillRun": TableSpec("Mill", ["mill_run_id"], "same_as_source", "One row per source mill record.", ["mill_run_id"]),
    "ARD_OPS_SalesOrder": TableSpec("Sales", ["order_no"], "unique_order_header", "One row per order header.", ["order_no"]),
    "ARD_OPS_SalesOrderLine": TableSpec("Sales", ["order_line_id"], "same_as_source", "One row per source sales line.", ["order_line_id"]),
    "ARD_OPS_WorkOrder": TableSpec("WorkOrder", ["wo_no"], "same_as_source", "One row per work order.", ["wo_no"], "ARD_OPS_WORKODER"),
    "ARD_OPS_BinCleaningLog": TableSpec("BinCleaning", ["cleaning_log_id"], "same_as_source", "One row per source bin-cleaning log.", ["cleaning_log_id"]),
    "ARD_OPS_FillOrder": TableSpec("Fills", ["fill_order_id"], "same_as_source", "One row per source fill record.", ["fill_order_id"]),
}

LOAD_ORDER = list(TABLE_SPECS.keys())


def utc_now_naive() -> datetime:
    return datetime.now(timezone.utc).replace(tzinfo=None)


def normalize_text(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return None
    return text


def normalize_upper(value) -> str | None:
    text = normalize_text(value)
    return text.upper() if text else None


def stable_bigint(*parts) -> int:
    text = "||".join("" if part is None else str(part) for part in parts)
    return int(hashlib.sha1(text.encode("utf-8")).hexdigest()[:15], 16)


def add_audit(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    now = utc_now_naive()
    out["created_date"] = now
    out["created_by"] = CREATED_BY
    out["updated_date"] = None
    out["updated_by"] = None
    return out


def assign_sequence_ids(df: pd.DataFrame, id_col: str, start: int) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out.insert(0, id_col, pd.array(range(start, start + len(out)), dtype="Int64"))
    return out


def load_source_excel(path: str | Path) -> dict[str, pd.DataFrame]:
    workbook = pd.read_excel(path, sheet_name=SOURCE_SHEETS)
    prepared: dict[str, pd.DataFrame] = {}
    for sheet_name, df in workbook.items():
        clean = df.copy()
        clean.columns = clean.columns.str.strip()
        clean.insert(0, "_source_row_num", range(2, len(clean) + 2))
        prepared[sheet_name] = clean
    return prepared


def parse_site_short_name(series: pd.Series) -> pd.DataFrame:
    clean = series.astype(str).str.strip()
    return pd.DataFrame(
        {
            "site_name": clean.str.split("-").str[0].str.strip().str.title(),
            "site_id": pd.to_numeric(clean.str.extract(r"-(\d+)$")[0], errors="coerce").astype("Int64"),
        }
    )


def add_unknown_dimension_row(df: pd.DataFrame, key_col: str, natural_col: str, extra: dict | None = None) -> pd.DataFrame:
    extra = extra or {}
    unknown = {col: None for col in df.columns}
    unknown[key_col] = 0
    unknown[natural_col] = UNKNOWN_TEXT
    unknown.update(extra)
    return pd.concat([pd.DataFrame([unknown]), df], ignore_index=True)


def assign_stable_id(df: pd.DataFrame, id_col: str, key_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    basis = out[key_cols].apply(
        lambda row: "||".join("" if pd.isna(v) else str(v) for v in row.tolist()),
        axis=1,
    )
    occurrence = basis.groupby(basis, dropna=False).cumcount().add(1)
    out[id_col] = [stable_bigint(b, occ) for b, occ in zip(basis, occurrence)]
    out[id_col] = out[id_col].astype("Int64")
    return out


def expected_count(source_df: pd.DataFrame, table_name: str) -> int:
    rule = TABLE_SPECS[table_name].count_rule
    if rule == "same_as_source":
        return len(source_df)
    if table_name == "ARD_OPS_SalesOrder":
        return source_df["ORDER_NO"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_Site":
        if "SITE_ID_OPS" in source_df.columns:
            return pd.to_numeric(source_df["SITE_ID_OPS"], errors="coerce").dropna().nunique()
    if table_name == "ARD_OPS_ItemClass":
        return source_df["ITEM_CLASS_DESC"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_Customer":
        return source_df["CUSTOMER_NM"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_ShipToAccount":
        return pd.to_numeric(source_df["ShipTo"], errors="coerce").dropna().nunique()
    if table_name == "ARD_OPS_Carrier":
        return source_df["Carrier_ID"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_ProductionMix":
        return source_df["ProductionMix"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_MaintenanceType":
        return source_df["MAINTENANCE_TYP"].fillna(UNKNOWN_TEXT).astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_CleaningType":
        return source_df["CleaningType_ID"].fillna(UNKNOWN_TEXT).astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_PackLine":
        work = source_df[["SITE_SHORT_NAME", "LINE"]].copy()
        site_bits = parse_site_short_name(work["SITE_SHORT_NAME"])
        work["site_id"] = site_bits["site_id"]
        work["line_name"] = work["LINE"].astype(str).str.strip().str.upper()
        return work.drop_duplicates(subset=["site_id", "line_name"]).shape[0]
    if table_name == "ARD_OPS_Bin":
        return source_df["Bin_ID"].astype(str).str.strip().nunique()
    if table_name == "ARD_OPS_Product":
        return -1
    return -1


def build_site(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    issue_rows: list[dict] = []

    parsed_frames = []
    for sheet in ["Pack", "Mill", "Sales", "WorkOrder"]:
        parsed = parse_site_short_name(raw[sheet]["SITE_SHORT_NAME"])
        parsed["ops_type"] = None
        parsed["region"] = None
        parsed["pack_plant"] = None
        parsed["company"] = None
        parsed["country_cd"] = None
        parsed["source_sheet"] = sheet
        parsed_frames.append(parsed)

    bin_sites = raw["BinCleaning"][
        ["SHORTPLANTNAME", "SITE_ID_OPS", "OPS_TYPE", "REGION", "PACKPLANT", "COMPANY", "COUNTRY_CD"]
    ].copy()
    bin_sites = bin_sites.rename(
        columns={
            "SHORTPLANTNAME": "site_name",
            "SITE_ID_OPS": "site_id",
            "OPS_TYPE": "ops_type",
            "REGION": "region",
            "PACKPLANT": "pack_plant",
            "COMPANY": "company",
            "COUNTRY_CD": "country_cd",
        }
    )
    bin_sites["site_id"] = pd.to_numeric(bin_sites["site_id"], errors="coerce").astype("Int64")
    bin_sites["source_sheet"] = "BinCleaning"
    parsed_frames.append(bin_sites)

    combined = pd.concat(parsed_frames, ignore_index=True)
    before = len(combined)
    combined = combined.dropna(subset=["site_id"]).copy()
    combined["site_name"] = combined["site_name"].apply(normalize_text)
    combined["site_name_upper"] = combined["site_name"].str.upper()
    combined = combined.sort_values(by=["site_id", "source_sheet"]).reset_index(drop=True)

    final_rows = []
    for site_id, grp in combined.groupby("site_id", dropna=False):
        site_name = grp["site_name"].dropna().iloc[0] if grp["site_name"].dropna().any() else None
        best = grp[grp["source_sheet"] == "BinCleaning"]
        row = (best.iloc[0] if not best.empty else grp.iloc[0]).to_dict()
        row["site_name"] = site_name
        final_rows.append(row)

    site_df = pd.DataFrame(final_rows)[["site_id", "site_name", "ops_type", "region", "pack_plant", "company", "country_cd"]]
    site_df = site_df.drop_duplicates(subset=["site_id"]).sort_values("site_id").reset_index(drop=True)
    site_df = add_audit(site_df)

    duplicate_rows = before - site_df.shape[0]
    if duplicate_rows:
        issue_rows.append(
            {
                "target_table": "ARD_OPS_Site",
                "source_sheet": "Multiple",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": duplicate_rows,
                "reason": "The target site dimension stores one row per site_id, so repeated source references collapse into a single site record.",
                "resolution": "Expected dimension behavior.",
            }
        )

    return site_df, issue_rows, pd.DataFrame()


def build_item_class(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    issues: list[dict] = []
    sales = raw["Sales"][["ITEM_CLASS_DESC"]].copy()
    sales["item_class_desc"] = sales["ITEM_CLASS_DESC"].apply(normalize_text)
    df = sales[["item_class_desc"]].drop_duplicates().reset_index(drop=True)
    df = df.dropna(subset=["item_class_desc"]).copy()
    df = df.sort_values("item_class_desc").reset_index(drop=True)
    df.insert(0, "item_class_id", pd.array(range(1, len(df) + 1), dtype="Int64"))

    duplicate_rows = len(sales) - sales["item_class_desc"].dropna().nunique()
    if duplicate_rows:
        issues.append(
            {
                "target_table": "ARD_OPS_ItemClass",
                "source_sheet": "Sales",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": duplicate_rows,
                "reason": "The target dimension stores one row per unique item class description.",
                "resolution": "Expected dimension behavior.",
            }
        )

    df = add_audit(df[["item_class_id", "item_class_desc"]].sort_values(["item_class_id"]).reset_index(drop=True))
    return df, issues, pd.DataFrame()


def build_product_stage(raw: dict[str, pd.DataFrame], item_class_df: pd.DataFrame):
    issues: list[dict] = []
    details: list[dict] = []

    # -------- SALES --------
    sales = raw["Sales"][["ITEM_NUM", "ITEM_DESC", "ITEM_CLASS_DESC"]].copy()
    sales["source_sheet"] = "Sales"

    sales["product_id"] = sales["ITEM_NUM"].apply(normalize_text)
    sales["product_desc"] = sales["ITEM_DESC"].apply(normalize_text)
    sales["item_class_desc"] = sales["ITEM_CLASS_DESC"].apply(normalize_text)

    # -------- FILLS --------
    fills = raw["Fills"][["_source_row_num", "Product", "Product_Desc"]].copy()
    fills["source_sheet"] = "Fills"

    fills["product_id"] = fills["Product"].apply(normalize_text)
    fills["product_desc"] = fills["Product_Desc"].apply(normalize_text)
    fills["item_class_desc"] = None

    # -------- COMBINE --------
    combined = pd.concat(
        [
            sales[["source_sheet", "product_id", "product_desc", "item_class_desc"]],
            fills[["source_sheet", "product_id", "product_desc", "item_class_desc"]],
        ],
        ignore_index=True,
    )

    # 🔥 CRITICAL FIX → normalize keys
    combined["product_id"] = (
        combined["product_id"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # remove null keys
    null_products = combined["product_id"].isna().sum()
    combined = combined.dropna(subset=["product_id"]).copy()

    # -------- DEDUP (SAFE) --------
    combined = combined.sort_values(by=["product_id", "source_sheet"]).reset_index(drop=True)
    deduped = combined.drop_duplicates(subset=["product_id"], keep="first").copy()

    # -------- MAP ITEM CLASS --------
    item_class_map = item_class_df.set_index("item_class_desc")["item_class_id"].to_dict()
    deduped["item_class_id"] = deduped["item_class_desc"].map(item_class_map).astype("Int64")

    # -------- ISSUES --------
    fills_only_missing = deduped["item_class_desc"].isna().sum()
    if fills_only_missing:
        issues.append({
            "target_table": "ARD_OPS_Product",
            "source_sheet": "Fills",
            "issue_type": "Missing item class left null",
            "impacted_rows": int(fills_only_missing),
            "reason": "Fills products do not include ITEM_CLASS_DESC.",
            "resolution": "item_class_id left null.",
        })

    duplicate_rows = len(combined) - len(deduped)
    if duplicate_rows:
        issues.append({
            "target_table": "ARD_OPS_Product",
            "source_sheet": "Sales/Fills",
            "issue_type": "Duplicate collapsed",
            "impacted_rows": duplicate_rows,
            "reason": "One row per product_id.",
            "resolution": "Expected behavior.",
        })

    if null_products:
        issues.append({
            "target_table": "ARD_OPS_Product",
            "source_sheet": "Sales/Fills",
            "issue_type": "Null product removed",
            "impacted_rows": int(null_products),
            "reason": "Null product_id not allowed.",
            "resolution": "Check source.",
        })

    out = deduped[["product_id", "product_desc", "item_class_id"]].copy()

    # 🔥 FINAL SAFETY (important)
    out = out.drop_duplicates(subset=["product_id"])

    return out, issues, pd.DataFrame(details)


def build_product(raw: dict[str, pd.DataFrame], item_class_df: pd.DataFrame):
    out, issues, detail_df = build_product_stage(raw, item_class_df)

    # 🔥 ensure clean again (double safety)
    out["product_id"] = (
        out["product_id"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    out = out.drop_duplicates(subset=["product_id"])

    out = out.sort_values("product_id").reset_index(drop=True)

    # sequence PK
    out = assign_sequence_ids(
        out[["product_id", "product_desc", "item_class_id"]].copy(),
        "product_pk",
        500
    )

    out = add_audit(out)

    return out, issues, detail_df


def build_customer(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    sales = raw["Sales"][["CUSTOMER_NM"]].copy()
    sales["customer_nm"] = sales["CUSTOMER_NM"].apply(normalize_text)
    deduped = sales.dropna(subset=["customer_nm"]).drop_duplicates(subset=["customer_nm"]).copy()
    deduped = deduped.sort_values("customer_nm").reset_index(drop=True)
    deduped.insert(0, "customer_id", pd.array(range(3000, 3000 + len(deduped)), dtype="Int64"))
    issues = []
    duplicate_rows = len(sales) - len(deduped)
    if duplicate_rows:
        issues.append(
            {
                "target_table": "ARD_OPS_Customer",
                "source_sheet": "Sales",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": duplicate_rows,
                "reason": "The customer dimension stores one row per customer name.",
                "resolution": "Expected dimension behavior.",
            }
        )
    out = add_audit(deduped[["customer_id", "customer_nm"]].sort_values("customer_nm").reset_index(drop=True))
    return out, issues, pd.DataFrame()


def build_ship_to(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    fills = raw["Fills"][
        ["ShipTo", "ShipToName", "SoldToName", "ShipTo_CITY", "ShipTo_STATE", "ShipTo_ZIP", "ShipTo_COUNTRY"]
    ].copy()
    fills["ship_to_id"] = pd.to_numeric(fills["ShipTo"], errors="coerce").astype("Int64")
    fills["ship_to_name"] = fills["ShipToName"].apply(normalize_text)
    fills["sold_to_name"] = fills["SoldToName"].apply(normalize_text)
    fills["city"] = fills["ShipTo_CITY"].apply(normalize_text)
    fills["state"] = fills["ShipTo_STATE"].apply(normalize_text)
    fills["zip"] = fills["ShipTo_ZIP"].apply(normalize_text)
    fills["country"] = fills["ShipTo_COUNTRY"].apply(normalize_text)
    null_ids = fills["ship_to_id"].isna().sum()
    deduped = fills.dropna(subset=["ship_to_id"]).drop_duplicates(subset=["ship_to_id"]).copy()
    issues = []
    if null_ids:
        issues.append(
            {
                "target_table": "ARD_OPS_ShipToAccount",
                "source_sheet": "Fills",
                "issue_type": "Null ship-to key removed",
                "impacted_rows": int(null_ids),
                "reason": "A null ShipTo cannot be loaded as a ship-to account.",
                "resolution": "Review source data if these rows are expected.",
            }
        )
    duplicate_rows = len(fills) - len(deduped) - null_ids
    if duplicate_rows:
        issues.append(
            {
                "target_table": "ARD_OPS_ShipToAccount",
                "source_sheet": "Fills",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": int(duplicate_rows),
                "reason": "The ship-to dimension stores one row per ShipTo identifier.",
                "resolution": "Expected dimension behavior.",
            }
        )
    out = add_audit(deduped[["ship_to_id", "ship_to_name", "sold_to_name", "city", "state", "zip", "country"]])
    return out, issues, pd.DataFrame()


def build_carrier(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    fills = raw["Fills"][["Carrier_ID"]].copy()
    fills["carrier_code"] = fills["Carrier_ID"].apply(normalize_text)
    deduped = fills.drop_duplicates(subset=["carrier_code"]).copy()
    deduped = deduped.dropna(subset=["carrier_code"])
    deduped = deduped.sort_values("carrier_code").reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["carrier_code"]], "carrier_id", 11000)
    issues = []
    if fills["carrier_code"].isna().sum():
        deduped = add_unknown_dimension_row(deduped[["carrier_id", "carrier_code"]], "carrier_id", "carrier_code")
        issues.append(
            {
                "target_table": "ARD_OPS_Carrier",
                "source_sheet": "Fills",
                "issue_type": "Unknown member inserted",
                "impacted_rows": int(fills["carrier_code"].isna().sum()),
                "reason": "Some Fills rows have no Carrier_ID, so an UNKNOWN member was added to keep those rows loadable.",
                "resolution": "Mapped missing carriers to UNKNOWN.",
            }
        )
    out = deduped[["carrier_id", "carrier_code"]]
    out = add_audit(out)
    return out, issues, pd.DataFrame()


def build_production_mix(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    mill = raw["Mill"][["ProductionMix"]].copy()
    mill["production_mix_code"] = mill["ProductionMix"].apply(normalize_text)
    deduped = mill.drop_duplicates(subset=["production_mix_code"]).dropna(subset=["production_mix_code"]).copy()
    deduped = deduped.sort_values("production_mix_code").reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["production_mix_code"]], "production_mix_id", 13000)
    if mill["production_mix_code"].isna().sum():
        deduped = add_unknown_dimension_row(deduped[["production_mix_id", "production_mix_code"]], "production_mix_id", "production_mix_code")
    out = deduped[["production_mix_id", "production_mix_code"]]
    out = add_audit(out)
    return out, [], pd.DataFrame()


def build_maintenance_type(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    work = raw["WorkOrder"][["_source_row_num", "MAINTENANCE_TYP"]].copy()
    work["maintenance_type"] = work["MAINTENANCE_TYP"].apply(normalize_text)
    null_count = work["maintenance_type"].isna().sum()
    detail_rows = work[work["maintenance_type"].isna()][["_source_row_num"]].copy()
    deduped = work[["maintenance_type"]].fillna(UNKNOWN_TEXT).drop_duplicates().copy()
    deduped = deduped.sort_values("maintenance_type").reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["maintenance_type"]], "maintenance_type_id", 15000)
    issues = []
    if null_count:
        issues.append(
            {
                "target_table": "ARD_OPS_MaintenanceType",
                "source_sheet": "WorkOrder",
                "issue_type": "Null maintenance type mapped to UNKNOWN",
                "impacted_rows": int(null_count),
                "reason": "Some work orders do not have MAINTENANCE_TYP populated.",
                "resolution": "Mapped those rows to the UNKNOWN dimension member so work orders still load.",
            }
        )
    out = add_audit(deduped[["maintenance_type_id", "maintenance_type"]].reset_index(drop=True))
    details = pd.DataFrame()
    if not detail_rows.empty:
        details = detail_rows.assign(
            target_table="ARD_OPS_MaintenanceType",
            source_sheet="WorkOrder",
            reason="Null MAINTENANCE_TYP was mapped to UNKNOWN.",
            source_key=lambda x: "Row " + x["_source_row_num"].astype(str),
            sample_values="MAINTENANCE_TYP = NULL",
        )[["target_table", "source_sheet", "reason", "source_key", "sample_values"]]
    return out, issues, details


def build_cleaning_type(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    cleaning = raw["BinCleaning"][["_source_row_num", "CleaningType_ID", "CleaningType", "CurrentCleaningStdFrequency"]].copy()
    cleaning["cleaning_type_id"] = cleaning["CleaningType_ID"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    cleaning["cleaning_type_desc"] = cleaning["CleaningType"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    null_count = (cleaning["CleaningType_ID"].apply(normalize_text).isna()).sum()
    deduped = cleaning.drop_duplicates(subset=["cleaning_type_id"]).copy()
    deduped = deduped.sort_values("cleaning_type_id").reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["cleaning_type_id", "cleaning_type_desc"]], "cleaning_type_pk", 17000)
    issues = []
    if null_count:
        issues.append(
            {
                "target_table": "ARD_OPS_CleaningType",
                "source_sheet": "BinCleaning",
                "issue_type": "Null cleaning type mapped to UNKNOWN",
                "impacted_rows": int(null_count),
                "reason": "Some bin-cleaning rows do not have CleaningType_ID populated.",
                "resolution": "Mapped those rows to the UNKNOWN dimension member so cleaning logs still load.",
            }
        )
    out = add_audit(deduped[["cleaning_type_pk", "cleaning_type_id", "cleaning_type_desc"]].reset_index(drop=True))
    details = pd.DataFrame()
    if null_count:
        details = cleaning[cleaning["CleaningType_ID"].apply(normalize_text).isna()][["_source_row_num"]].assign(
            target_table="ARD_OPS_CleaningType",
            source_sheet="BinCleaning",
            reason="Null CleaningType_ID was mapped to UNKNOWN.",
            source_key=lambda x: "Row " + x["_source_row_num"].astype(str),
            sample_values="CleaningType_ID = NULL",
        )[["target_table", "source_sheet", "reason", "source_key", "sample_values"]]
    return out, issues, details


def build_pack_line(raw: dict[str, pd.DataFrame], site_df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    pack = raw["Pack"][["SITE_SHORT_NAME", "LINE"]].copy()
    parsed = parse_site_short_name(pack["SITE_SHORT_NAME"])
    pack["site_id"] = parsed["site_id"]
    pack["line_name"] = pack["LINE"].apply(normalize_upper)
    deduped = pack.drop_duplicates(subset=["site_id", "line_name"]).copy()
    deduped = deduped.sort_values(["site_id", "line_name"]).reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["line_name", "site_id"]], "line_id", 20000)
    issues = []
    duplicate_rows = len(pack) - len(deduped)
    if duplicate_rows:
        issues.append(
            {
                "target_table": "ARD_OPS_PackLine",
                "source_sheet": "Pack",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": int(duplicate_rows),
                "reason": "The target pack-line dimension stores one row per site_id and line_name combination.",
                "resolution": "Expected dimension behavior.",
            }
        )
    out = add_audit(deduped[["line_id", "line_name", "site_id"]].reset_index(drop=True))
    return out, issues, pd.DataFrame()


def build_bin(raw: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    bc = raw["BinCleaning"][["Bin_ID", "BinPurpose", "SITE_ID_OPS"]].copy()
    bc["bin_id"] = bc["Bin_ID"].apply(normalize_text)
    bc["bin_purpose"] = bc["BinPurpose"].apply(normalize_text)
    bc["site_id"] = pd.to_numeric(bc["SITE_ID_OPS"], errors="coerce").astype("Int64")
    deduped = bc.drop_duplicates(subset=["bin_id"]).dropna(subset=["bin_id"]).copy()
    deduped = deduped.sort_values("bin_id").reset_index(drop=True)
    deduped = assign_sequence_ids(deduped[["bin_id", "bin_purpose", "site_id"]], "bin_pk", 41000)
    issues = []
    duplicate_rows = len(bc) - len(deduped)
    if duplicate_rows:
        issues.append(
            {
                "target_table": "ARD_OPS_Bin",
                "source_sheet": "BinCleaning",
                "issue_type": "Duplicate source grain collapsed",
                "impacted_rows": int(duplicate_rows),
                "reason": "The target bin dimension stores one row per bin_id.",
                "resolution": "Expected dimension behavior.",
            }
        )
    out = add_audit(deduped[["bin_pk", "bin_id", "bin_purpose", "site_id"]].reset_index(drop=True))
    return out, issues, pd.DataFrame()


def build_pack_run(raw: dict[str, pd.DataFrame], site_df: pd.DataFrame, pack_line_df: pd.DataFrame, product_lookup: set[str]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    pack = raw["Pack"].copy()
    site_bits = parse_site_short_name(pack["SITE_SHORT_NAME"])
    pack["site_id"] = site_bits["site_id"]
    pack["product_id"] = pack["PRODUCT"].apply(normalize_text)
    unmapped_mask = ~pack["product_id"].isin(product_lookup)
    unmapped_mask = unmapped_mask & pack["product_id"].notna()
    pack["line_name"] = pack["LINE"].apply(normalize_upper)
    line_map = pack_line_df.assign(_key=pack_line_df["site_id"].astype(str) + "|" + pack_line_df["line_name"].astype(str)).set_index("_key")["line_id"].to_dict()
    pack["_line_key"] = pack["site_id"].astype(str) + "|" + pack["line_name"].astype(str)
    pack["line_id"] = pack["_line_key"].map(line_map).astype("Int64")
    pack["pack_date"] = pd.to_datetime(pack["PACKDATE"], errors="coerce")
    pack["good_units"] = pd.to_numeric(pack["GoodUnits"], errors="coerce")
    pack["target_units"] = pd.to_numeric(pack["TargetUnits"], errors="coerce")
    pack["total_units"] = pd.to_numeric(pack["TotalUnits"], errors="coerce").astype("Int64")
    pack["calc_dt"] = pd.to_numeric(pack["CalcDT"], errors="coerce")
    pack["minutes_run"] = pd.to_numeric(pack["MinutesRun"], errors="coerce")
    pack["pack_oee"] = pd.to_numeric(pack["Pack OEE"], errors="coerce")
    pack = pack.sort_values("_source_row_num").reset_index(drop=True)
    pack = assign_sequence_ids(pack, "pack_run_id", 23000)
    issues = []
    if pack["line_id"].isna().sum():
        issues.append(
            {
                "target_table": "ARD_OPS_PackRun",
                "source_sheet": "Pack",
                "issue_type": "Missing line lookup",
                "impacted_rows": int(pack["line_id"].isna().sum()),
                "reason": "Pack rows with an unmapped site_id and line_name cannot resolve a line_id.",
                "resolution": "Review SITE_SHORT_NAME and LINE values in source data.",
            }
        )
    if pack["product_id"].isna().sum():
        issues.append(
            {
                "target_table": "ARD_OPS_PackRun",
                "source_sheet": "Pack",
                "issue_type": "Missing product lookup",
                "impacted_rows": int(pack["product_id"].isna().sum()),
                "reason": "Pack rows with a product code not found in the product dimension cannot resolve product_id.",
                "resolution": "Review PRODUCT values in the Pack sheet.",
            }
        )
    missing_products = int(unmapped_mask.sum())
    if missing_products:
        issues.append(
            {
                "target_table": "ARD_OPS_PackRun",
                "source_sheet": "Pack",
                "issue_type": "Unmapped product id",
                "impacted_rows": missing_products,
                "reason": "Some Pack PRODUCT values do not exist in the 197-row product dimension from Sales/Fills.",
                "resolution": "Those pack-only codes are set to NULL in PackRun so the Oracle foreign key does not fail.",
            }
        )
        pack.loc[unmapped_mask, "product_id"] = None
    out = add_audit(
        pack[
            ["pack_run_id", "site_id", "product_id", "line_id", "pack_date", "good_units", "target_units", "total_units", "calc_dt", "minutes_run", "pack_oee"]
        ]
    )
    return out, issues, pd.DataFrame()


def build_mill_run(raw: dict[str, pd.DataFrame], production_mix_df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    mill = raw["Mill"].copy()
    site_bits = parse_site_short_name(mill["SITE_SHORT_NAME"])
    mill["site_id"] = site_bits["site_id"]
    mix_map = production_mix_df.set_index("production_mix_code")["production_mix_id"].to_dict()
    mill["production_mix_code"] = mill["ProductionMix"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    mill["production_mix_id"] = mill["production_mix_code"].map(mix_map).astype("Int64")
    mill["mill_date"] = pd.to_datetime(mill["MILLDATE"], errors="coerce")
    mill["unit"] = mill["UNIT"].apply(normalize_text)
    mill["calc_downtime"] = pd.to_numeric(mill["Calculated Downtime"], errors="coerce")
    mill["min_run"] = pd.to_numeric(mill["MinRun"], errors="coerce")
    mill["no_demand_downtime"] = pd.to_numeric(mill["NODemand Downtime"], errors="coerce")
    mill["mill_oee"] = pd.to_numeric(mill["Mill OEE"], errors="coerce")
    mill = mill.sort_values("_source_row_num").reset_index(drop=True)
    mill = assign_sequence_ids(mill, "mill_run_id", 26000)
    out = add_audit(
        mill[["mill_run_id", "site_id", "mill_date", "unit", "production_mix_id", "calc_downtime", "min_run", "no_demand_downtime", "mill_oee"]]
    )
    return out, [], pd.DataFrame()


def build_sales_order(raw: dict[str, pd.DataFrame], customer_df: pd.DataFrame, product_lookup: set[str]) -> tuple[pd.DataFrame, pd.DataFrame, list[dict], pd.DataFrame]:
    sales = raw["Sales"].copy()
    site_bits = parse_site_short_name(sales["SITE_SHORT_NAME"])
    sales["site_id"] = site_bits["site_id"]
    cust_map = customer_df.set_index("customer_nm")["customer_id"].to_dict()
    sales["customer_nm"] = sales["CUSTOMER_NM"].apply(normalize_text)
    sales["customer_id"] = sales["customer_nm"].map(cust_map).astype("Int64")
    sales["order_no"] = sales["ORDER_NO"].apply(normalize_text)
    sales["product_id"] = sales["ITEM_NUM"].apply(normalize_text) 
    sales["ship_date"] = pd.to_datetime(sales["SHIP_DATE"], errors="coerce")
    sales["order_status"] = sales["ORDER_STATUS_INDICATOR"].apply(normalize_text)
    sales["invoice_cwts"] = pd.to_numeric(sales["INVOICE_CWTS"], errors="coerce")

    order_df = sales[["order_no", "site_id", "customer_id", "ship_date", "order_status"]].drop_duplicates(subset=["order_no"]).copy()
    order_df = order_df.sort_values("order_no").reset_index(drop=True)
    order_df = assign_sequence_ids(order_df, "order_id", 45000)
    order_df = add_audit(order_df)

    line_df = sales[["_source_row_num", "order_no", "product_id", "invoice_cwts"]].copy()
    line_df = line_df.sort_values("_source_row_num").reset_index(drop=True)
    line_df = assign_sequence_ids(line_df, "order_line_id", 30000)
    line_df = add_audit(line_df[["order_line_id", "order_no", "product_id", "invoice_cwts"]])

    issues = [
        {
            "target_table": "ARD_OPS_SalesOrder",
            "source_sheet": "Sales",
            "issue_type": "Header rows collapsed from detail",
            "impacted_rows": int(len(sales) - len(order_df)),
            "reason": "SalesOrder is a header table, so multiple line rows with the same ORDER_NO become one order header.",
            "resolution": "Expected OLTP normalization behavior.",
        }
    ]
    return order_df, line_df, issues, pd.DataFrame()


def build_work_order(raw: dict[str, pd.DataFrame], maintenance_type_df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    work = raw["WorkOrder"].copy()
    site_bits = parse_site_short_name(work["SITE_SHORT_NAME"])
    work["site_id"] = site_bits["site_id"]
    mt_map = maintenance_type_df.set_index("maintenance_type")["maintenance_type_id"].to_dict()
    work["maintenance_type"] = work["MAINTENANCE_TYP"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    work["maintenance_type_id"] = work["maintenance_type"].map(mt_map).astype("Int64")
    work["wo_no"] = work["WO_NO"].apply(normalize_text)
    work["category_cd"] = work["CATEGORY_CD"].apply(normalize_text)
    work["status_cd"] = work["STATUS_CD"].apply(normalize_text)
    work["preventive_corrective_ind"] = work["PREVENTIVE_CORRECTIVE_IND"].apply(normalize_text)
    work["late_indicator"] = work["LATE_INDICATOR"].apply(normalize_text)
    work["required_date"] = pd.to_datetime(work["REQUIRED_DATE"], errors="coerce")
    for col in ["WO_COUNT", "WO_LATE_COUNT", "WO_ONTIME_COUNT", "WO_UPCOMING_COUNT"]:
        work[col] = pd.to_numeric(work[col], errors="coerce").astype("Int64")
    out = work[
        [
            "wo_no",
            "site_id",
            "maintenance_type_id",
            "category_cd",
            "status_cd",
            "preventive_corrective_ind",
            "late_indicator",
            "required_date",
            "WO_COUNT",
            "WO_LATE_COUNT",
            "WO_ONTIME_COUNT",
            "WO_UPCOMING_COUNT",
        ]
    ].rename(
        columns={
            "WO_COUNT": "wo_count",
            "WO_LATE_COUNT": "wo_late_count",
            "WO_ONTIME_COUNT": "wo_ontime_count",
            "WO_UPCOMING_COUNT": "wo_upcoming_count",
        }
    )
    out = out.sort_values("wo_no").reset_index(drop=True)
    out = assign_sequence_ids(out, "workorder_pk", 48000)
    out = add_audit(out)
    return out, [], pd.DataFrame()


def build_bin_cleaning_log(raw: dict[str, pd.DataFrame], cleaning_type_df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    bc = raw["BinCleaning"].copy()
    bc["bin_id"] = bc["Bin_ID"].apply(normalize_text)
    bc["cleaning_type_id"] = bc["CleaningType_ID"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    bc["cleaning_completed_on"] = pd.to_datetime(bc["CleaningCompletedOn"], errors="coerce")
    bc["cleaning_completed_by"] = bc["CleaningCompletedBy"].apply(normalize_text)
    bc["days_since_last_cleaning"] = pd.to_numeric(bc["DaysSinceLastCleaning"], errors="coerce")
    bc["bin_status"] = bc["BinStatus"].apply(normalize_text)
    bc["clean_standard_in_place"] = pd.to_numeric(bc["CleanStandardInPlace"], errors="coerce")
    bc["clean_standard_freq"] = pd.to_numeric(bc["CurrentCleaningStdFrequency"], errors="coerce")
    bc["comments"] = bc["comments"].apply(normalize_text)
    bc["last_refresh_time"] = pd.to_datetime(bc["LastRefreshTime"], errors="coerce")
    bc["status_as_of_date"] = pd.to_datetime(bc["Status_AsOf_Date"], errors="coerce")
    bc["status_as_of_wk_start"] = pd.to_datetime(bc["Status_AsOf_Date_1stofweek"], errors="coerce")
    bc = bc.sort_values("_source_row_num").reset_index(drop=True)
    bc = assign_sequence_ids(bc, "cleaning_log_id", 35000)
    out = add_audit(
        bc[
            [
                "cleaning_log_id",
                "bin_id",
                "cleaning_type_id",
                "cleaning_completed_on",
                "cleaning_completed_by",
                "days_since_last_cleaning",
                "bin_status",
                "clean_standard_in_place",
                "clean_standard_freq",
                "comments",
                "last_refresh_time",
                "status_as_of_date",
                "status_as_of_wk_start",
            ]
        ]
    )
    return out, [], pd.DataFrame()


def build_fill_order(raw: dict[str, pd.DataFrame], ship_to_df: pd.DataFrame, carrier_df: pd.DataFrame, product_lookup: set[str]) -> tuple[pd.DataFrame, list[dict], pd.DataFrame]:
    fills = raw["Fills"].copy()
    site_map = {"ALBANY": 1001, "AYER": 1004, "OGDEN": 1025}
    fills["site_id"] = fills["Site_Name"].apply(normalize_upper).map(site_map).astype("Int64")
    carrier_map = carrier_df.set_index("carrier_code")["carrier_id"].to_dict()
    fills["carrier_code"] = fills["Carrier_ID"].apply(normalize_text).fillna(UNKNOWN_TEXT)
    fills["carrier_id"] = fills["carrier_code"].map(carrier_map).astype("Int64")
    fills["product_id"] = fills["Product"].apply(normalize_text)
    fills["order_number"] = pd.to_numeric(fills["OrderNumber"], errors="coerce").astype("Int64")
    fills["vessel_id"] = pd.to_numeric(fills["Vessel_ID"], errors="coerce").astype("Int64")
    fills["ship_to_id"] = pd.to_numeric(fills["ShipTo"], errors="coerce").astype("Int64")
    fills["bulk_or_sack"] = fills["BulkOrSack"].apply(normalize_text)
    fills["miles"] = pd.to_numeric(fills["Miles"], errors="coerce").astype("Int64")
    fills["drop_ship"] = fills["Drop_Ship"].apply(normalize_text)
    fills["modifier"] = fills["modifier"].apply(normalize_text)
    fills["delivery_note_no"] = pd.to_numeric(fills["DeliveryNoteNo"], errors="coerce").astype("Int64")
    fills["load_date"] = pd.to_datetime(fills["Load_Date"], errors="coerce")
    fills["ship_date"] = pd.to_datetime(fills["Ship_Date"], errors="coerce")
    fills["released_date"] = pd.to_datetime(fills["Released_Date"], errors="coerce")
    for col in ["ShippedCwt", "Volume", "exceptioncwts", "Baseline", "MaxForLoad", "GoalForLoad", "VAR_to_Goal", "VAR_to_Baseline", "VAR_to_LoadMax", "VAR_to_SiteMax"]:
        fills[col] = pd.to_numeric(fills[col], errors="coerce")
    for col in ["SiteGoaL", "SiteMax", "MadeGoal"]:
        fills[col] = pd.to_numeric(fills[col], errors="coerce").astype("Int64")
    fills["excluded"] = fills["Excluded"].apply(normalize_text)
    fills["exception_flag"] = fills["Exception_flag"].apply(normalize_text)
    fills = fills.sort_values("_source_row_num").reset_index(drop=True)
    fills = assign_sequence_ids(fills, "fill_order_id", 38000)
    out = add_audit(
        fills[
            [
                "fill_order_id",
                "order_number",
                "vessel_id",
                "site_id",
                "ship_to_id",
                "carrier_id",
                "product_id",
                "bulk_or_sack",
                "miles",
                "drop_ship",
                "modifier",
                "delivery_note_no",
                "load_date",
                "ship_date",
                "released_date",
                "ShippedCwt",
                "Volume",
                "SiteGoaL",
                "SiteMax",
                "exceptioncwts",
                "Baseline",
                "MaxForLoad",
                "GoalForLoad",
                "MadeGoal",
                "VAR_to_Goal",
                "VAR_to_Baseline",
                "VAR_to_LoadMax",
                "VAR_to_SiteMax",
                "excluded",
                "exception_flag",
            ]
        ].rename(
            columns={
                "ShippedCwt": "shipped_cwt",
                "Volume": "volume",
                "SiteGoaL": "site_goal",
                "SiteMax": "site_max",
                "exceptioncwts": "exception_cwts",
                "Baseline": "baseline",
                "MaxForLoad": "max_for_load",
                "GoalForLoad": "goal_for_load",
                "MadeGoal": "made_goal",
                "VAR_to_Goal": "var_to_goal",
                "VAR_to_Baseline": "var_to_baseline",
                "VAR_to_LoadMax": "var_to_load_max",
                "VAR_to_SiteMax": "var_to_site_max",
            }
        )
    )
    return out, [], pd.DataFrame()


def build_all_tables(raw: dict[str, pd.DataFrame]) -> tuple[dict[str, pd.DataFrame], dict[str, list[dict]], dict[str, pd.DataFrame]]:
    tables: dict[str, pd.DataFrame] = {}
    issues: dict[str, list[dict]] = {}
    details: dict[str, pd.DataFrame] = {}

    tables["ARD_OPS_Site"], issues["ARD_OPS_Site"], details["ARD_OPS_Site"] = build_site(raw)
    tables["ARD_OPS_ItemClass"], issues["ARD_OPS_ItemClass"], details["ARD_OPS_ItemClass"] = build_item_class(raw)
    product_stage, issues["ARD_OPS_Product"], details["ARD_OPS_Product"] = build_product_stage(raw, tables["ARD_OPS_ItemClass"])
    product_lookup = set(product_stage["product_id"].dropna().tolist())
    tables["ARD_OPS_Product"], _, _ = build_product(raw, tables["ARD_OPS_ItemClass"])
    tables["ARD_OPS_Customer"], issues["ARD_OPS_Customer"], details["ARD_OPS_Customer"] = build_customer(raw)
    tables["ARD_OPS_ShipToAccount"], issues["ARD_OPS_ShipToAccount"], details["ARD_OPS_ShipToAccount"] = build_ship_to(raw)
    tables["ARD_OPS_Carrier"], issues["ARD_OPS_Carrier"], details["ARD_OPS_Carrier"] = build_carrier(raw)
    tables["ARD_OPS_ProductionMix"], issues["ARD_OPS_ProductionMix"], details["ARD_OPS_ProductionMix"] = build_production_mix(raw)
    tables["ARD_OPS_MaintenanceType"], issues["ARD_OPS_MaintenanceType"], details["ARD_OPS_MaintenanceType"] = build_maintenance_type(raw)
    tables["ARD_OPS_CleaningType"], issues["ARD_OPS_CleaningType"], details["ARD_OPS_CleaningType"] = build_cleaning_type(raw)
    tables["ARD_OPS_PackLine"], issues["ARD_OPS_PackLine"], details["ARD_OPS_PackLine"] = build_pack_line(raw, tables["ARD_OPS_Site"])
    tables["ARD_OPS_Bin"], issues["ARD_OPS_Bin"], details["ARD_OPS_Bin"] = build_bin(raw)
    tables["ARD_OPS_PackRun"], issues["ARD_OPS_PackRun"], details["ARD_OPS_PackRun"] = build_pack_run(raw, tables["ARD_OPS_Site"], tables["ARD_OPS_PackLine"], product_lookup)
    tables["ARD_OPS_MillRun"], issues["ARD_OPS_MillRun"], details["ARD_OPS_MillRun"] = build_mill_run(raw, tables["ARD_OPS_ProductionMix"])
    sales_order, sales_line, sales_issues, sales_details = build_sales_order(raw, tables["ARD_OPS_Customer"], product_lookup)
    tables["ARD_OPS_SalesOrder"] = sales_order
    tables["ARD_OPS_SalesOrderLine"] = sales_line
    issues["ARD_OPS_SalesOrder"] = sales_issues
    issues["ARD_OPS_SalesOrderLine"] = []
    details["ARD_OPS_SalesOrder"] = sales_details
    details["ARD_OPS_SalesOrderLine"] = pd.DataFrame()
    tables["ARD_OPS_WorkOrder"], issues["ARD_OPS_WorkOrder"], details["ARD_OPS_WorkOrder"] = build_work_order(raw, tables["ARD_OPS_MaintenanceType"])
    tables["ARD_OPS_BinCleaningLog"], issues["ARD_OPS_BinCleaningLog"], details["ARD_OPS_BinCleaningLog"] = build_bin_cleaning_log(raw, tables["ARD_OPS_CleaningType"])
    tables["ARD_OPS_FillOrder"], issues["ARD_OPS_FillOrder"], details["ARD_OPS_FillOrder"] = build_fill_order(raw, tables["ARD_OPS_ShipToAccount"], tables["ARD_OPS_Carrier"], product_lookup)

    return tables, issues, details


def summary_dataframe(raw: dict[str, pd.DataFrame], tables: dict[str, pd.DataFrame], issues: dict[str, list[dict]]) -> pd.DataFrame:
    rows = []
    source_map = {
        "ARD_OPS_Site": raw["BinCleaning"],
        "ARD_OPS_ItemClass": raw["Sales"],
        "ARD_OPS_Product": pd.concat([raw["Sales"][["ITEM_NUM"]].rename(columns={"ITEM_NUM": "product_id"}), raw["Fills"][["Product"]].rename(columns={"Product": "product_id"})], ignore_index=True),
        "ARD_OPS_Customer": raw["Sales"],
        "ARD_OPS_ShipToAccount": raw["Fills"],
        "ARD_OPS_Carrier": raw["Fills"],
        "ARD_OPS_ProductionMix": raw["Mill"],
        "ARD_OPS_MaintenanceType": raw["WorkOrder"],
        "ARD_OPS_CleaningType": raw["BinCleaning"],
        "ARD_OPS_PackLine": raw["Pack"],
        "ARD_OPS_Bin": raw["BinCleaning"],
        "ARD_OPS_PackRun": raw["Pack"],
        "ARD_OPS_MillRun": raw["Mill"],
        "ARD_OPS_SalesOrder": raw["Sales"],
        "ARD_OPS_SalesOrderLine": raw["Sales"],
        "ARD_OPS_WorkOrder": raw["WorkOrder"],
        "ARD_OPS_BinCleaningLog": raw["BinCleaning"],
        "ARD_OPS_FillOrder": raw["Fills"],
    }
    for table_name in LOAD_ORDER:
        source_df = source_map[table_name]
        target_df = tables[table_name]
        spec = TABLE_SPECS[table_name]
        exp = expected_count(source_df, table_name)
        exp_text = exp if exp >= 0 else "Derived from combined Sales/Fills product grain"
        rows.append(
            {
                "target_table": table_name,
                "source_sheet": spec.source_sheet,
                "source_rows": len(source_df),
                "expected_target_rows": exp_text,
                "actual_target_rows": len(target_df),
                "business_key": ", ".join(spec.business_key),
                "grain_description": spec.grain_description,
                "status": "PASS" if (exp == len(target_df) or exp < 0) else "CHECK",
                "top_reason": issues.get(table_name, [{}])[0].get("reason", "Counts align with expected target grain.") if issues.get(table_name) else "Counts align with expected target grain.",
            }
        )
    return pd.DataFrame(rows)


def build_test_cases(raw: dict[str, pd.DataFrame], tables: dict[str, pd.DataFrame], issues: dict[str, list[dict]]) -> pd.DataFrame:
    tests = []
    for table_name, df in tables.items():
        spec = TABLE_SPECS[table_name]
        key_cols = spec.business_key
        null_key_count = int(df[key_cols].isna().any(axis=1).sum())
        dup_key_count = int(df.duplicated(subset=key_cols, keep=False).sum())
        audit_nulls = int(df[["created_date", "created_by"]].isna().any(axis=1).sum())
        source_df = raw["Sales"] if spec.source_sheet == "Sales/Fills" else raw.get(spec.source_sheet.split("/")[0], pd.DataFrame())
        exp = expected_count(source_df, table_name) if not source_df.empty else -1
        count_result = "PASS" if exp < 0 or exp == len(df) else "CHECK"

        tests.extend(
            [
                {
                    "test_case_id": f"{table_name}_TC01",
                    "target_table": table_name,
                    "test_case": "Business key null count",
                    "result": "PASS" if null_key_count == 0 else "FAIL",
                    "actual_value": null_key_count,
                    "expected_value": 0,
                    "explanation": "Business key columns should not be null in the transformed target dataset.",
                },
                {
                    "test_case_id": f"{table_name}_TC02",
                    "target_table": table_name,
                    "test_case": "Business key duplicate count",
                    "result": "PASS" if dup_key_count == 0 else "FAIL",
                    "actual_value": dup_key_count,
                    "expected_value": 0,
                    "explanation": "The transformed target grain should be unique on the table business key.",
                },
                {
                    "test_case_id": f"{table_name}_TC03",
                    "target_table": table_name,
                    "test_case": "Expected source-to-target row count",
                    "result": count_result,
                    "actual_value": len(df),
                    "expected_value": exp if exp >= 0 else "See explanation",
                    "explanation": spec.grain_description,
                },
                {
                    "test_case_id": f"{table_name}_TC04",
                    "target_table": table_name,
                    "test_case": "Audit columns populated",
                    "result": "PASS" if audit_nulls == 0 else "FAIL",
                    "actual_value": audit_nulls,
                    "expected_value": 0,
                    "explanation": "created_date and created_by should be populated for every target row.",
                },
                {
                    "test_case_id": f"{table_name}_TC05",
                    "target_table": table_name,
                    "test_case": "Transformation issues review",
                    "result": "CHECK" if issues.get(table_name) else "PASS",
                    "actual_value": len(issues.get(table_name, [])),
                    "expected_value": 0,
                    "explanation": "CHECK means the workbook contains a documented source-to-target explanation for this table.",
                },
            ]
        )
    return pd.DataFrame(tests)


def build_issues_dataframe(issues: dict[str, list[dict]]) -> pd.DataFrame:
    rows = []
    for table_issues in issues.values():
        rows.extend(table_issues)
    return pd.DataFrame(rows)


def build_detail_dataframe(details: dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = [df for df in details.values() if df is not None and not df.empty]
    if not frames:
        return pd.DataFrame(columns=["target_table", "source_sheet", "reason", "source_key", "sample_values"])
    return pd.concat(frames, ignore_index=True)


def write_validation_workbook(output_path: str | Path, raw: dict[str, pd.DataFrame], tables: dict[str, pd.DataFrame], issues: dict[str, list[dict]], details: dict[str, pd.DataFrame]) -> Path:
    output = Path(output_path)
    summary_df = summary_dataframe(raw, tables, issues)
    tests_df = build_test_cases(raw, tables, issues)
    issues_df = build_issues_dataframe(issues)
    details_df = build_detail_dataframe(details)

    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        tests_df.to_excel(writer, sheet_name="Test_Cases", index=False)
        issues_df.to_excel(writer, sheet_name="Issues", index=False)
        details_df.to_excel(writer, sheet_name="Issue_Details", index=False)

        for table_name, df in tables.items():
            preview = df.head(25)
            preview.to_excel(writer, sheet_name=table_name.replace("ARD_OPS_", "")[:31], index=False)

    return output


def make_oracle_dsn(cfg: dict) -> str:
    if cfg.get("dsn"):
        return cfg["dsn"]
    host = cfg["host"]
    port = cfg.get("port", 1521)
    service_name = cfg.get("service_name")
    sid = cfg.get("sid")
    if service_name:
        return f"{host}:{port}/{service_name}"
    if sid:
        return f"{host}:{port}/{sid}"
    raise ValueError("Oracle config must include either dsn, service_name, or sid.")


def open_oracle_connection(cfg: dict) -> oracledb.Connection:
    connect_kwargs = {
        "user": cfg["username"],
        "password": cfg["password"],
        "dsn": make_oracle_dsn(cfg),
    }
    if cfg.get("thick_mode"):
        try:
            oracledb.init_oracle_client()
        except Exception:
            pass
    return oracledb.connect(**connect_kwargs)


def test_oracle_connection(cfg: dict) -> tuple[bool, str]:
    try:
        with open_oracle_connection(cfg) as conn:
            with conn.cursor() as cursor:
                cursor.execute("SELECT 1 FROM dual")
                cursor.fetchone()
        return True, "Oracle connection OK"
    except Exception as exc:
        return False, str(exc)


def _safe_db_value(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            return None
        return value.to_pydatetime()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    return value
def build_oracle_merge_sql(table_name: str, all_cols: list[str], key_cols: list[str]) -> str:
    """
    Safe Oracle MERGE builder.

    Fixes:
    - removes surrogate PK columns from INSERT/UPDATE
    - uses proper NULL-safe comparison (no TO_CHAR)
    """

    # ❌ remove surrogate PK columns (very important)
    non_pk_cols = [c for c in all_cols if not c.endswith("_PK")]

    select_clause = ", ".join(f":{col} {col}" for col in non_pk_cols)

    # ✅ NULL-safe comparison (cleaner + faster than TO_CHAR)
    on_clause = " AND ".join(
        f"(t.{col} = s.{col} OR (t.{col} IS NULL AND s.{col} IS NULL))"
        for col in key_cols
    )

    # ✅ update only non-key, non-PK columns
    update_cols = [
        col for col in non_pk_cols
        if col not in key_cols
    ]

    sql = [
        f"MERGE INTO {table_name} t",
        f"USING (SELECT {select_clause} FROM dual) s",
        f"ON ({on_clause})",
    ]

    if update_cols:
        update_clause = ", ".join(f"t.{col} = s.{col}" for col in update_cols)
        sql.append(f"WHEN MATCHED THEN UPDATE SET {update_clause}")

    # ✅ insert WITHOUT PK
    insert_cols = ", ".join(non_pk_cols)
    insert_vals = ", ".join(f"s.{col}" for col in non_pk_cols)

    sql.append(
        f"WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})"
    )

    return "\n".join(sql)

def upsert_dataframe(df: pd.DataFrame, table_name: str, key_cols: list[str], cfg: dict):
    import oracledb

    conn = oracledb.connect(**cfg)

    # 🚨 remove duplicates based on TRUE key
    df = df.drop_duplicates(subset=key_cols).copy()

    # ❌ remove PK column completely
    cols = [c for c in df.columns if not c.endswith("_PK")]

    update_cols = [c for c in cols if c not in key_cols]

    merge_sql = f"""
        MERGE INTO {table_name} t
        USING (
            SELECT {', '.join([f':{i+1} {c}' for i, c in enumerate(cols)])}
            FROM dual
        ) s
        ON ({' AND '.join([f't.{k} = s.{k}' for k in key_cols])})

        WHEN MATCHED THEN
            UPDATE SET {', '.join([f't.{c} = s.{c}' for c in update_cols])}

        WHEN NOT MATCHED THEN
            INSERT ({', '.join(cols)})
            VALUES ({', '.join([f's.{c}' for c in cols])})
    """

    bind_rows = [tuple(row) for row in df[cols].itertuples(index=False, name=None)]

    try:
        with conn.cursor() as cursor:
            cursor.executemany(merge_sql, bind_rows)
        conn.commit()
        print(f"✅ {table_name} loaded: {len(df)} rows")

    except Exception as e:
        print(f"❌ FAILED TABLE: {table_name}")
        print(f"👉 key_cols used: {key_cols}")
        raise

    finally:
        conn.close()
def run_single_table(tables: dict[str, pd.DataFrame], table_name: str, cfg: dict) -> None:
    
    # ✅ check table exists
    if table_name not in tables:
        print(f"❌ {table_name} not found in tables dict")
        return

    df = tables[table_name]

    # ✅ check empty
    if df is None or df.empty:
        print(f"⚠️ {table_name} is empty, skipping load")
        return

    # ✅ get spec
    spec = TABLE_SPECS[table_name]
    physical_name = spec.physical_table_name or table_name.upper()
    key_cols = spec.loader_key

    # 🚨 remove duplicates BEFORE load (CRITICAL)
    before = len(df)
    df = df.drop_duplicates(subset=key_cols, keep="first").copy()
    after = len(df)

    if before != after:
        print(f"⚠️ removed {before - after} duplicate rows in {table_name} based on {key_cols}")

    # ✅ load
    try:
        upsert_dataframe(df, physical_name, key_cols, cfg)
        print(f"✅ {physical_name} loaded successfully: {len(df)} rows")

    except Exception as e:
        print(f"❌ error loading {physical_name}: {str(e)}")
        raise


def run_all_tables(tables: dict[str, pd.DataFrame], cfg: dict) -> None:
    for table_name in LOAD_ORDER:
        print(f"Loading {table_name} ...")
        run_single_table(tables, table_name, cfg)


def preview_table(tables: dict[str, pd.DataFrame], table_name: str, rows: int = 5) -> pd.DataFrame:
    return tables[table_name].head(rows)


def export_diagnostics_json(output_path: str | Path, raw: dict[str, pd.DataFrame], tables: dict[str, pd.DataFrame], issues: dict[str, list[dict]]) -> Path:
    payload = {
        "generated_at_utc": utc_now_naive().isoformat(),
        "summary": summary_dataframe(raw, tables, issues).to_dict(orient="records"),
        "issues": build_issues_dataframe(issues).to_dict(orient="records"),
    }
    target = Path(output_path)
    target.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return target


## Cell 3 — Oracle Configuration  ⚠️ EDIT BEFORE RUNNING

In [114]:
# ──────────────────────────────────────────────────────────────
#  EDIT THESE VALUES BEFORE RUNNING
# ──────────────────────────────────────────────────────────────

EXCEL_FILE_PATH = Path(r'c:\Users\Maruthi R M\Downloads\Ardent_Mills_Data.xlsx')
VALIDATION_WORKBOOK_PATH = Path('Ardent_Mills_ETL_Test_Cases.xlsx')
DIAGNOSTICS_JSON_PATH = Path('Ardent_Mills_ETL_Diagnostics.json')

ORACLE_CONFIG = {
    'host': 'ec2-3-111-0-185.ap-south-1.compute.amazonaws.com',
    'port': 1521,
    'service_name': 'orcl',
    'username': 'maruthi_nov25',
    'password': 'maruthi_nov25',
    # optional: 'dsn': 'host:port/service_name',
    # optional: 'thick_mode': True,
}
# ──────────────────────────────────────────────────────────────

## Cell 4 — Oracle Connection Test

In [115]:
ok, message = test_oracle_connection(ORACLE_CONFIG)
print('Oracle connection status:', ok)
print(message)

Oracle connection status: True
Oracle connection OK


## Cell 5 — Load Excel Sheets

In [116]:
raw = load_source_excel(EXCEL_FILE_PATH)
print({k: v.shape for k, v in raw.items()})

{'Pack': (333, 12), 'Mill': (367, 9), 'Sales': (1611, 10), 'WorkOrder': (628, 13), 'BinCleaning': (162, 22), 'Fills': (491, 45)}


## Cell 6 — Transform: ARD_OPS_Site

In [117]:
df_site, issues_site, details_site = build_site(raw)
print('ARD_OPS_Site:', df_site.shape)
df_site.head(10)

ARD_OPS_Site: (3, 11)


,site_id,site_name,ops_type,region,pack_plant,company,country_cd,created_date,created_by,updated_date,updated_by
0,1001,Albany,Mill,East,Pack,AMUS,US,2026-04-27 10:39:32.982591,Maruthi_R_M,None,None
1,1004,Ayer,Mill,East,No_Pack,AMUS,US,2026-04-27 10:39:32.982591,Maruthi_R_M,None,None
2,1025,Ogden,Mill,West,Pack,AMUS,US,2026-04-27 10:39:32.982591,Maruthi_R_M,None,None


## Cell 7 — Transform: ARD_OPS_ItemClass

In [118]:
df_item_class, issues_item_class, details_item_class = build_item_class(raw)
print('ARD_OPS_ItemClass:', df_item_class.shape)
df_item_class.head(10)

ARD_OPS_ItemClass: (7, 6)


,item_class_id,item_class_desc,created_date,created_by,updated_date,updated_by
0,1,BRAN/GERM/NUFIBER,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
1,2,HARD FLOUR,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
2,3,ORGANIC,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
3,4,RED WHOLE WHEAT,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
4,5,SOFT FLOUR,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
5,6,WHEAT BYPRODUCTS,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None
6,7,WHITE WHOLE WHEAT,2026-04-27 10:39:33.075904,Maruthi_R_M,None,None


## Cell 8 — Transform: ARD_OPS_Product

In [119]:
df_product, issues_product, details_product = build_product(raw, df_item_class)
print('ARD_OPS_Product:', df_product.shape)
df_product

ARD_OPS_Product: (198, 8)


,product_pk,product_id,product_desc,item_class_id,created_date,created_by,updated_date,updated_by
0,500,06-000001,WHEAT MIDDLINGS LOOSE-BULK,6,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
1,501,06-000006,WHEAT MIDDLINGS RED DOG-BULK,6,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
2,502,374327345,WHEAT MIDDLING LOOSE-BULK123,2,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
3,503,5100146,2ND CLEAR FLR BULK-AA(FEED),<NA>,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
4,504,5100214,2ND CLEAR FLR BULK-AA-R51604(FEED),<NA>,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
...,...,...,...,...,...,...,...,...
193,693,5165519,GENERAL BRANDS PASTRY FLR 50LB-RA (HT),5,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
194,694,5165666,NORTHEAST FLR BULK-FD,<NA>,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
195,695,5165699,PAISANOS PREMIUM PAT FLR 50LB-RK,2,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None
196,696,5165969,GREAT VALUE AP FLR 4/5LB-RG,2,2026-04-27 10:39:33.387828,Maruthi_R_M,None,None


## Cell 9 — Transform: ARD_OPS_Customer

In [120]:
df_customer, issues_customer, details_customer = build_customer(raw)
print('ARD_OPS_Customer:', df_customer.shape)
df_customer

ARD_OPS_Customer: (146, 6)


,customer_id,customer_nm,created_date,created_by,updated_date,updated_by
0,3000,A B P CORP,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
1,3001,A FIORILLO CO LLC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
2,3002,A FODERA & SON INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
3,3003,A OLIVERI & SONS INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
4,3004,AMARALS BAKERY INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
...,...,...,...,...,...,...
141,3141,WHEN PIGS FLY INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
142,3142,WHITE OAK MILLS INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
143,3143,WILLOW RUN FOODS INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None
144,3144,WONTON FOOD INC,2026-04-27 10:39:33.463508,Maruthi_R_M,None,None


## Cell 10 — Transform: ARD_OPS_ShipToAccount

In [121]:
df_ship_to, issues_ship_to, details_ship_to = build_ship_to(raw)
print('ARD_OPS_ShipToAccount:', df_ship_to.shape)
df_ship_to.head(10)

ARD_OPS_ShipToAccount: (74, 11)


,ship_to_id,ship_to_name,sold_to_name,city,state,zip,country,created_date,created_by,updated_date,updated_by
0,1500004425,"DOMINOS PIZZA LLC - EAST GRANDBY, C",DOMINOS PIZZA LLC - ANN ARBOR MI,EAST GRANBY,CT,06026-9718,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
4,1000015124,IT`LL BE PIZZA,IT`LL BE PIZZA,SCARBOROUGH,ME,04074-9783,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
6,2000008612,BIMBO HUNGRIA ALBANY,BIMBO HUNGRIA CO,ALBANY,NY,12206-2229,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
8,2500000026,TREEHOUSE - FROZEN,TREEHOUSE PRIVATE BRANDS,OGDEN,UT,84404-1342,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
9,20038136,CARAVAN,CARAVAN INGREDIENTS,East Rutherford,NJ,7073,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
15,1500046317,MAXIMS NUTRICARE INC W JORDON,MAXIMS NUTRICARE INC W JORDON,WEST JORDAN,UT,84081-5694,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
16,2500008564,PEPPERIDGE FARM INC RICHMOND,PEPPERIDGE FARM INC,RICHMOND,UT,84333-1499,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
24,1500046809,AMERICAN NUTRITION INC,AMERICAN NUTRITION INC,OGDEN,UT,84401-3529,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
29,2500069064,AUTOMATIC ROLLS OF NEW EN,NORTHEAST FOODS INCORPORA,DAYVILLE,CT,06241-1537,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None
37,2500076873,VERMONT BREAD CO,KOFFEE KUP BAKERY INCORPO,BRATTLEBORO,VT,05301-8681,US,2026-04-27 10:39:33.531739,Maruthi_R_M,None,None


## Cell 11 — Transform: ARD_OPS_Carrier

In [122]:
df_carrier, issues_carrier, details_carrier = build_carrier(raw)
print('ARD_OPS_Carrier:', df_carrier.shape)
df_carrier.head(10)

ARD_OPS_Carrier: (2, 6)


,carrier_id,carrier_code,created_date,created_by,updated_date,updated_by
0,11000,FOLW,2026-04-27 10:39:33.600948,Maruthi_R_M,None,None
1,11001,WWSP,2026-04-27 10:39:33.600948,Maruthi_R_M,None,None


## Cell 12 — Transform: ARD_OPS_ProductionMix

In [123]:
df_production_mix, issues_production_mix, details_production_mix = build_production_mix(raw)
print('ARD_OPS_ProductionMix:', df_production_mix.shape)
df_production_mix.head(10)

ARD_OPS_ProductionMix: (21, 6)


,production_mix_id,production_mix_code,created_date,created_by,updated_date,updated_by
0,13000,AW42 CA46,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
1,13001,AW48,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
2,13002,AW48 CA56,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
3,13003,D54 CA50,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
4,13004,D58,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
5,13005,E45,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
6,13006,F48 CA56,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
7,13007,F50,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
8,13008,H50,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None
9,13009,H54,2026-04-27 10:39:33.661748,Maruthi_R_M,None,None


## Cell 13 — Transform: ARD_OPS_MaintenanceType

In [124]:
df_maintenance_type, issues_maintenance_type, details_maintenance_type = build_maintenance_type(raw)
print('ARD_OPS_MaintenanceType:', df_maintenance_type.shape)
df_maintenance_type.head(10)

ARD_OPS_MaintenanceType: (9, 6)


,maintenance_type_id,maintenance_type,created_date,created_by,updated_date,updated_by
0,15000,ASSET FAILURE / BREAKDOWN,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
1,15001,CAPITAL PROJECT,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
2,15002,CORRECTIVE TASK,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
3,15003,FOLLOW-UP TASK,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
4,15004,LUBRICATION,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
5,15005,PREDICTIVE TASK,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
6,15006,PREVENTATIVE TASK,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
7,15007,ROUTINE TASK,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None
8,15008,UNKNOWN,2026-04-27 10:39:33.728800,Maruthi_R_M,None,None


## Cell 14 — Transform: ARD_OPS_CleaningType

In [125]:
df_cleaning_type, issues_cleaning_type, details_cleaning_type = build_cleaning_type(raw)
print('ARD_OPS_CleaningType:', df_cleaning_type.shape)
df_cleaning_type.head(10)

ARD_OPS_CleaningType: (4, 7)


,cleaning_type_pk,cleaning_type_id,cleaning_type_desc,created_date,created_by,updated_date,updated_by
0,17000,Entry,interior entry,2026-04-27 10:39:33.802439,Maruthi_R_M,None,None
1,17001,Light,flashlight insp,2026-04-27 10:39:33.802439,Maruthi_R_M,None,None
2,17002,UNKNOWN,UNKNOWN,2026-04-27 10:39:33.802439,Maruthi_R_M,None,None
3,17003,Whip,air hose whippi,2026-04-27 10:39:33.802439,Maruthi_R_M,None,None


## Cell 15 — Transform: ARD_OPS_PackLine

In [126]:
df_pack_line, issues_pack_line, details_pack_line = build_pack_line(raw, df_site)
print('ARD_OPS_PackLine:', df_pack_line.shape)
df_pack_line.head(10)

ARD_OPS_PackLine: (3, 7)


,line_id,line_name,site_id,created_date,created_by,updated_date,updated_by
0,20000,B&B VALVE,1001,2026-04-27 10:39:33.889048,Maruthi_R_M,None,None
1,20001,ITAL,1025,2026-04-27 10:39:33.889048,Maruthi_R_M,None,None
2,20002,PREMIER TECH,1025,2026-04-27 10:39:33.889048,Maruthi_R_M,None,None


## Cell 16 — Transform: ARD_OPS_Bin

In [127]:
df_bin, issues_bin, details_bin = build_bin(raw)
print('ARD_OPS_Bin:', df_bin.shape)
df_bin.head(10)

ARD_OPS_Bin: (162, 8)


,bin_pk,bin_id,bin_purpose,site_id,created_date,created_by,updated_date,updated_by
0,41000,1AJK-22,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
1,41001,1AJK-23,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
2,41002,1AJK-24,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
3,41003,1AJK-25,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
4,41004,1AJK-26,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
5,41005,1AJK-27,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
6,41006,1AJK-28,Specialty Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
7,41007,1AJK-29,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
8,41008,1AJK-30,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None
9,41009,1AJK-31,Flour Storage,1004,2026-04-27 10:39:34.025489,Maruthi_R_M,None,None


## Cell 17 — Transform: ARD_OPS_PackRun

In [128]:
product_lookup = set(df_product['product_id'].dropna().tolist())
df_pack_run, issues_pack_run, details_pack_run = build_pack_run(raw, df_site, df_pack_line, product_lookup)
print('ARD_OPS_PackRun:', df_pack_run.shape)
df_pack_run.head(10)

ARD_OPS_PackRun: (333, 15)


,pack_run_id,site_id,product_id,line_id,pack_date,good_units,target_units,total_units,calc_dt,minutes_run,pack_oee,created_date,created_by,updated_date,updated_by
0,23000,1001,5101204,20000,2017-03-10,450.0,1024.0,454,17.93750,32.0,0.439453,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
1,23001,1001,5102879,20000,2017-03-10,1082.0,1760.0,1088,21.18750,55.0,0.614773,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
2,23002,1001,5103937,20000,2017-03-10,911.0,2048.0,1023,35.53125,64.0,0.444824,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
3,23003,1001,5104949,20000,2017-03-10,723.0,2240.0,730,47.40625,70.0,0.322768,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
4,23004,1001,5108382,20000,2017-03-10,950.0,1376.0,952,13.31250,43.0,0.690407,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
5,23005,1001,5109899,20000,2017-03-10,150.0,352.0,150,6.31250,11.0,0.426136,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
6,23006,1001,5111475,20000,2017-03-10,1033.0,2400.0,1046,42.71875,75.0,0.430417,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
7,23007,1001,5115222,20000,2017-03-10,500.0,832.0,501,10.37500,26.0,0.600962,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
8,23008,1001,5118034,20000,2017-03-10,3537.0,6656.0,3562,97.46875,208.0,0.531400,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None
9,23009,1001,5118888,20000,2017-03-10,456.0,704.0,458,7.75000,22.0,0.647727,2026-04-27 10:39:34.147624,Maruthi_R_M,None,None


## Cell 18 — Transform: ARD_OPS_MillRun

In [129]:
df_mill_run, issues_mill_run, details_mill_run = build_mill_run(raw, df_production_mix)
print('ARD_OPS_MillRun:', df_mill_run.shape)
df_mill_run.head(10)

ARD_OPS_MillRun: (367, 13)


,mill_run_id,site_id,mill_date,unit,production_mix_id,calc_downtime,min_run,no_demand_downtime,mill_oee,created_date,created_by,updated_date,updated_by
0,26000,1004,2017-03-15,A,13009,4.971176,327.0,0.0,0.984798,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
1,26001,1004,2017-03-15,A,13008,9.427941,1113.0,0.0,0.991529,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
2,26002,1004,2017-03-15,B,13011,-5.563448,1440.0,0.0,1.003864,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
3,26003,1004,2017-03-14,A,13008,1.900588,345.0,0.0,0.994491,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
4,26004,1004,2017-03-14,B,13011,-28.433793,345.0,0.0,1.082417,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
5,26005,1004,2017-03-14,A,13013,1095.000000,0.0,1095.0,0.000000,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
6,26006,1004,2017-03-14,B,13013,1095.000000,0.0,1095.0,0.000000,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
7,26007,1004,2017-03-17,A,13008,47.585294,1440.0,0.0,0.966955,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
8,26008,1004,2017-03-17,B,13013,360.000000,0.0,360.0,0.000000,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None
9,26009,1004,2017-03-17,B,13011,43.072759,1080.0,0.0,0.960118,2026-04-27 10:39:34.264971,Maruthi_R_M,None,None


## Cell 19 — Transform: ARD_OPS_SalesOrder And ARD_OPS_SalesOrderLine

In [130]:
df_sales_order, df_sales_order_line, issues_sales_order, details_sales_order = build_sales_order(raw, df_customer, product_lookup)
issues_sales_order_line = []
details_sales_order_line = pd.DataFrame()
print('ARD_OPS_SalesOrder:', df_sales_order.shape)
print('ARD_OPS_SalesOrderLine:', df_sales_order_line.shape)
df_sales_order.head(10)
df_sales_order_line.head(10)


ARD_OPS_SalesOrder: (1326, 10)
ARD_OPS_SalesOrderLine: (1611, 8)


,order_line_id,order_no,product_id,invoice_cwts,created_date,created_by,updated_date,updated_by
0,30000,82378825,5122073,450.0,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
1,30001,M0060739,06-000001,600.0,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
2,30002,M0291253,06-000001,578.4,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
3,30003,M0289716,06-000001,579.0,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
4,30004,M0291254,06-000001,659.4,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
5,30005,M0289717,06-000001,598.4,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
6,30006,82387215,5162628,520.6,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
7,30007,82375234,5113229,1000.0,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
8,30008,82387978,5162628,498.8,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None
9,30009,82378446,5127113,425.0,2026-04-27 10:39:34.400695,Maruthi_R_M,None,None


## Cell 20 — Transform: ARD_OPS_WorkOrder

In [131]:
df_work_order, issues_work_order, details_work_order = build_work_order(raw, df_maintenance_type)
print('ARD_OPS_WorkOrder:', df_work_order.shape)
df_work_order.head(10)

ARD_OPS_WorkOrder: (628, 17)


,workorder_pk,wo_no,site_id,maintenance_type_id,category_cd,status_cd,preventive_corrective_ind,late_indicator,required_date,wo_count,wo_late_count,wo_ontime_count,wo_upcoming_count,created_date,created_by,updated_date,updated_by
0,48000,WO-100028100,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-05,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
1,48001,WO-100037483,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-07,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
2,48002,WO-100041578,1001,15006,Asset Maintenance Orders,SCHEDULED,CORRECTIVE,UPCOMING,2017-07-15,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
3,48003,WO-100045525,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-16,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
4,48004,WO-100052190,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-15,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
5,48005,WO-100055161,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-28,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
6,48006,WO-100055877,1004,15006,Asset Maintenance Orders,COMPLETED,PREVENTIVE,ON_TIME,2017-03-19,1,0,1,0,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
7,48007,WO-100056779,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-28,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
8,48008,WO-100059315,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-17,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None
9,48009,WO-100059316,1004,15001,Capital Work Orders,TO PROCESS,CORRECTIVE,UPCOMING,2017-05-17,1,0,0,1,2026-04-27 10:39:34.478470,Maruthi_R_M,None,None


## Cell 21 — Transform: ARD_OPS_BinCleaningLog

In [132]:
df_bin_cleaning_log, issues_bin_cleaning_log, details_bin_cleaning_log = build_bin_cleaning_log(raw, df_cleaning_type)
print('ARD_OPS_BinCleaningLog:', df_bin_cleaning_log.shape)
df_bin_cleaning_log.head(10)

ARD_OPS_BinCleaningLog: (162, 17)


,cleaning_log_id,bin_id,cleaning_type_id,cleaning_completed_on,cleaning_completed_by,days_since_last_cleaning,bin_status,clean_standard_in_place,clean_standard_freq,comments,last_refresh_time,status_as_of_date,status_as_of_wk_start,created_date,created_by,updated_date,updated_by
0,35000,1AJK-22,Light,2017-03-03,David Dean,20.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
1,35001,1AJK-23,Light,2017-03-08,David Dean,15.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
2,35002,1AJK-24,Light,2017-03-16,David Dean,7.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
3,35003,1AJK-25,Light,2017-03-16,David Dean,7.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
4,35004,1AJK-26,Light,2017-03-08,Kerry Henry,15.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
5,35005,1AJK-27,Light,2017-03-16,David Dean,7.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
6,35006,1AJK-28,Light,2017-02-22,Kerry Henry,29.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
7,35007,1AJK-29,Light,2017-02-22,David Dean,29.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
8,35008,1AJK-30,Light,2017-03-08,David Dean,15.0,OnTime,30.0,30,air horned,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None
9,35009,1AJK-31,Light,2017-03-08,David Dean,15.0,OnTime,30.0,30,None,2017-03-23 03:24:31.763,2017-03-23 11:47:25.153,2017-03-19,2026-04-27 10:39:34.568074,Maruthi_R_M,None,None


## Cell 22 — Transform: ARD_OPS_FillOrder

In [133]:
df_fill_order, issues_fill_order, details_fill_order = build_fill_order(raw, df_ship_to, df_carrier, product_lookup)
print('ARD_OPS_FillOrder:', df_fill_order.shape)
df_fill_order.head(10)

ARD_OPS_FillOrder: (491, 34)


,fill_order_id,order_number,vessel_id,site_id,ship_to_id,carrier_id,product_id,bulk_or_sack,miles,drop_ship,...,var_to_goal,var_to_baseline,var_to_load_max,var_to_site_max,excluded,exception_flag,created_date,created_by,updated_date,updated_by
0,38000,82380810,505046,1001,1500004425,11000,5138429,B,0,N,...,-1.0,16.0,-31.0,-31.0,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
1,38001,82380813,506019,1001,1500004425,11000,5138429,B,0,N,...,-6.0,11.0,-36.0,-36.0,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
2,38002,82380815,509030,1001,1500004425,11000,5138429,B,0,N,...,0.0,17.0,-30.0,-30.0,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
3,38003,82390126,510002,1001,1500004425,11000,5138429,B,0,N,...,9.0,26.0,-21.0,-21.0,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
4,38004,82375215,505076,1004,1000015124,11000,5124446,B,0,N,...,7.5,17.5,-22.5,-22.5,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
5,38005,82375214,507038,1004,1000015124,11000,5124446,B,0,N,...,4.0,14.0,-26.0,-26.0,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
6,38006,82383367,514023,1001,2000008612,11000,5108999,B,0,N,...,11.0,11.0,11.0,-29.0,NotExcluded,Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
7,38007,82383369,505048,1001,2000008612,11000,5108999,B,0,N,...,11.0,11.0,11.0,-29.0,NotExcluded,Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
8,38008,82382756,962019,1025,2500000026,11001,5162628,B,0,N,...,3.2,32.2,-26.8,-26.8,NotExcluded,No Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None
9,38009,82370161,505054,1001,20038136,11000,5123614,B,0,N,...,14.0,14.0,14.0,-66.0,NotExcluded,Exception,2026-04-27 10:39:34.645749,Maruthi_R_M,None,None


## Cell 23 — Build Tables Registry

In [134]:
tables = {
    'ARD_OPS_Site': df_site,
    'ARD_OPS_ItemClass': df_item_class,
    'ARD_OPS_Product': df_product,
    'ARD_OPS_Customer': df_customer,
    'ARD_OPS_ShipToAccount': df_ship_to,
    'ARD_OPS_Carrier': df_carrier,
    'ARD_OPS_ProductionMix': df_production_mix,
    'ARD_OPS_MaintenanceType': df_maintenance_type,
    'ARD_OPS_CleaningType': df_cleaning_type,
    'ARD_OPS_PackLine': df_pack_line,
    'ARD_OPS_Bin': df_bin,
    'ARD_OPS_PackRun': df_pack_run,
    'ARD_OPS_MillRun': df_mill_run,
    'ARD_OPS_SalesOrder': df_sales_order,
    'ARD_OPS_SalesOrderLine': df_sales_order_line,
    'ARD_OPS_WorkOrder': df_work_order,
    'ARD_OPS_BinCleaningLog': df_bin_cleaning_log,
    'ARD_OPS_FillOrder': df_fill_order,
}
issues = {
    'ARD_OPS_Site': issues_site,
    'ARD_OPS_ItemClass': issues_item_class,
    'ARD_OPS_Product': issues_product,
    'ARD_OPS_Customer': issues_customer,
    'ARD_OPS_ShipToAccount': issues_ship_to,
    'ARD_OPS_Carrier': issues_carrier,
    'ARD_OPS_ProductionMix': issues_production_mix,
    'ARD_OPS_MaintenanceType': issues_maintenance_type,
    'ARD_OPS_CleaningType': issues_cleaning_type,
    'ARD_OPS_PackLine': issues_pack_line,
    'ARD_OPS_Bin': issues_bin,
    'ARD_OPS_PackRun': issues_pack_run,
    'ARD_OPS_MillRun': issues_mill_run,
    'ARD_OPS_SalesOrder': issues_sales_order,
    'ARD_OPS_SalesOrderLine': issues_sales_order_line,
    'ARD_OPS_WorkOrder': issues_work_order,
    'ARD_OPS_BinCleaningLog': issues_bin_cleaning_log,
    'ARD_OPS_FillOrder': issues_fill_order,
}
details = {
    'ARD_OPS_Site': details_site,
    'ARD_OPS_ItemClass': details_item_class,
    'ARD_OPS_Product': details_product,
    'ARD_OPS_Customer': details_customer,
    'ARD_OPS_ShipToAccount': details_ship_to,
    'ARD_OPS_Carrier': details_carrier,
    'ARD_OPS_ProductionMix': details_production_mix,
    'ARD_OPS_MaintenanceType': details_maintenance_type,
    'ARD_OPS_CleaningType': details_cleaning_type,
    'ARD_OPS_PackLine': details_pack_line,
    'ARD_OPS_Bin': details_bin,
    'ARD_OPS_PackRun': details_pack_run,
    'ARD_OPS_MillRun': details_mill_run,
    'ARD_OPS_SalesOrder': details_sales_order,
    'ARD_OPS_SalesOrderLine': details_sales_order_line,
    'ARD_OPS_WorkOrder': details_work_order,
    'ARD_OPS_BinCleaningLog': details_bin_cleaning_log,
    'ARD_OPS_FillOrder': details_fill_order,
}
summary_df = summary_dataframe(raw, tables, issues)
summary_df

,target_table,source_sheet,source_rows,expected_target_rows,actual_target_rows,business_key,grain_description,status,top_reason
0,ARD_OPS_Site,BinCleaning,162,3,3,site_id,One row per site_id.,PASS,The target site dimension stores one row per s...
1,ARD_OPS_ItemClass,Sales,1611,7,7,item_class_id,One row per item class.,PASS,The target dimension stores one row per unique...
2,ARD_OPS_Product,Sales/Fills,2102,Derived from combined Sales/Fills product grain,198,product_id,One row per product.,PASS,Fills products do not include ITEM_CLASS_DESC.
3,ARD_OPS_Customer,Sales,1611,146,146,customer_id,One row per customer.,PASS,The customer dimension stores one row per cust...
4,ARD_OPS_ShipToAccount,Fills,491,74,74,ship_to_id,One row per ship-to account.,PASS,The ship-to dimension stores one row per ShipT...
5,ARD_OPS_Carrier,Fills,491,2,2,carrier_id,One row per carrier.,PASS,Counts align with expected target grain.
6,ARD_OPS_ProductionMix,Mill,367,21,21,production_mix_id,One row per production mix code.,PASS,Counts align with expected target grain.
7,ARD_OPS_MaintenanceType,WorkOrder,628,9,9,maintenance_type_id,One row per maintenance type.,PASS,Some work orders do not have MAINTENANCE_TYP p...
8,ARD_OPS_CleaningType,BinCleaning,162,4,4,cleaning_type_id,One row per cleaning type.,PASS,Some bin-cleaning rows do not have CleaningTyp...
9,ARD_OPS_PackLine,Pack,333,3,3,line_id,One row per site and line.,PASS,The target pack-line dimension stores one row ...


## Cell 24 — Export Validation Workbook

In [135]:
write_validation_workbook(VALIDATION_WORKBOOK_PATH, raw, tables, issues, details)
export_diagnostics_json(DIAGNOSTICS_JSON_PATH, raw, tables, issues)
print('Validation workbook created at', VALIDATION_WORKBOOK_PATH.resolve())
print('Diagnostics JSON created at', DIAGNOSTICS_JSON_PATH.resolve())

Validation workbook created at C:\Users\Maruthi R M\Downloads\Ardent_Mills_ETL_Test_Cases.xlsx
Diagnostics JSON created at C:\Users\Maruthi R M\Downloads\Ardent_Mills_ETL_Diagnostics.json


## Cell 25 — Load Single Table To Oracle

In [138]:
# Example: change table name and run one table at a time
run_single_table(tables, 'ARD_OPS_Site', ORACLE_CONFIG)

❌ error loading ARD_OPS_SITE: connect() got an unexpected keyword argument 'username'


TypeError: connect() got an unexpected keyword argument 'username'

## Cell 26 — Load All Tables Incrementally

In [139]:
run_all_tables(tables, ORACLE_CONFIG)

Loading ARD_OPS_Site ...
❌ error loading ARD_OPS_SITE: connect() got an unexpected keyword argument 'username'


TypeError: connect() got an unexpected keyword argument 'username'

## Cell 27 — Verify Oracle Row Counts

In [ ]:
TABLES = [
    'ARD_OPS_SITE', 'ARD_OPS_ITEMCLASS', 'ARD_OPS_PRODUCT', 'ARD_OPS_CUSTOMER',
    'ARD_OPS_SHIPTOACCOUNT', 'ARD_OPS_CARRIER', 'ARD_OPS_PRODUCTIONMIX',
    'ARD_OPS_MAINTENANCETYPE', 'ARD_OPS_CLEANINGTYPE', 'ARD_OPS_PACKLINE',
    'ARD_OPS_BIN', 'ARD_OPS_PACKRUN', 'ARD_OPS_MILLRUN', 'ARD_OPS_SALESORDER',
    'ARD_OPS_SALESORDERLINE', 'ARD_OPS_WORKODER', 'ARD_OPS_BINCLEANINGLOG',
    'ARD_OPS_FILLORDER'
]

counts = []
with open_oracle_connection(ORACLE_CONFIG) as conn:
    with conn.cursor() as cursor:
        for t in TABLES:
            try:
                cursor.execute(f'SELECT COUNT(*) FROM {t}')
                counts.append((t, cursor.fetchone()[0], 'OK'))
            except Exception as e:
                counts.append((t, 'ERROR', str(e)))

df_counts = pd.DataFrame(counts, columns=['Table', 'Row Count', 'Status'])
print(df_counts.to_string(index=False))

                  Table  Row Count Status
           ARD_OPS_SITE          3     OK
      ARD_OPS_ITEMCLASS          7     OK
        ARD_OPS_PRODUCT        197     OK
       ARD_OPS_CUSTOMER        145     OK
  ARD_OPS_SHIPTOACCOUNT         74     OK
        ARD_OPS_CARRIER          2     OK
  ARD_OPS_PRODUCTIONMIX         21     OK
ARD_OPS_MAINTENANCETYPE          9     OK
   ARD_OPS_CLEANINGTYPE          4     OK
       ARD_OPS_PACKLINE          3     OK
            ARD_OPS_BIN        162     OK
        ARD_OPS_PACKRUN        333     OK
        ARD_OPS_MILLRUN        367     OK
     ARD_OPS_SALESORDER       1325     OK
 ARD_OPS_SALESORDERLINE       1610     OK
       ARD_OPS_WORKODER        628     OK
 ARD_OPS_BINCLEANINGLOG        162     OK
      ARD_OPS_FILLORDER        491     OK


In [ ]:
3+7+197+145+74+2+21+9+4+3+162+333+367+1325+1610+628+162+491

5543

In [ ]:
import pandas as pd

# =====================================
# 1. LOAD DATA
# =====================================
file_path = "Ardent_Mills_Data.xlsx"
data = pd.read_excel(file_path, sheet_name=None)

pack = data["Pack"]
mill = data["Mill"]
sales = data["Sales"]
work = data["WorkOrder"]
bin_clean = data["BinCleaning"]
fills = data["Fills"]


In [ ]:

# =====================================
# 2. COMMON CLEAN FUNCTION
# =====================================
def clean(x):
    if pd.isna(x):
        return None
    return str(x).strip()


In [ ]:

# =====================================
# 3. SITE TABLE
# =====================================
site = bin_clean[["SITE_ID_OPS", "SHORTPLANTNAME"]].copy()
site["site_id"] = pd.to_numeric(site["SITE_ID_OPS"], errors="coerce")
site["site_name"] = site["SHORTPLANTNAME"].apply(clean)

site = site.dropna(subset=["site_id"]).drop_duplicates("site_id")
site = site[["site_id", "site_name"]].reset_index(drop=True)
site


,site_id,site_name
0,1004,Ayer
1,1001,Albany
2,1025,Ogden


In [ ]:

# =====================================
# 4. PRODUCT TABLE
# =====================================
prod_sales = sales[["ITEM_NUM", "ITEM_DESC"]].rename(
    columns={"ITEM_NUM": "product_id", "ITEM_DESC": "product_desc"}
)

prod_fills = fills[["Product", "Product_Desc"]].rename(
    columns={"Product": "product_id", "Product_Desc": "product_desc"}
)

product = pd.concat([prod_sales, prod_fills])
product["product_id"] = product["product_id"].apply(clean)

product = product.dropna(subset=["product_id"])
product = product.drop_duplicates("product_id").reset_index(drop=True)
product.insert(0, "product_pk", range(1, len(product)+1))
product


,product_pk,product_id,product_desc
0,1,5122073,ROSE HG FLR 50LB-RK7 ...
1,2,06-000001,WHEAT MIDDLINGS LOOSE-BULK ...
2,3,5162628,WHITE SPRAY PASTRY FLR BULK-CB ...
3,4,5113229,FOOD GRADE SOFT WHITE BRAN BULK-AA ...
4,5,5127113,WHITE SNOW FLR 50LB-RA ...
...,...,...,...
192,193,5163181,HO TOY NOODLE FLR 50LB-BI3 ...
193,194,5123568,SPRING HEARTH FLR 50LB-RG ...
194,195,5115571,(21094) MUFFIN FLR BULK-AA-NWF ...
195,196,5100304,(21067) 42 ASH HARD FLR BULK-AA-NWF ...


In [ ]:

# =====================================
# 5. CUSTOMER TABLE
# =====================================
customer = sales[["CUSTOMER_NM"]].copy()
customer["customer_nm"] = customer["CUSTOMER_NM"].apply(clean)

customer = customer.dropna().drop_duplicates().reset_index(drop=True)
customer.insert(0, "customer_id", range(1000, 1000+len(customer)))
customer

,customer_id,CUSTOMER_NM,customer_nm
0,1000,ORLANDO FOODS INC ...,ORLANDO FOODS INC
1,1001,LARKIN CATTLE CO ...,LARKIN CATTLE CO
2,1002,TREEHOUSE PRIVATE BRANDS ...,TREEHOUSE PRIVATE BRANDS
3,1003,THURSTON FOODS INC ...,THURSTON FOODS INC
4,1004,S AND D MORREALE INC ...,S AND D MORREALE INC
...,...,...,...
140,1140,NEWLY WEDS BREADING INC ...,NEWLY WEDS BREADING INC
141,1141,HO TOY NOODLES INC ...,HO TOY NOODLES INC
142,1142,A B P CORP ...,A B P CORP
143,1143,SEMICAN INTERNATIONAL ...,SEMICAN INTERNATIONAL


In [ ]:

# =====================================
# 6. CARRIER TABLE
# =====================================
carrier = fills[["Carrier_ID"]].copy()
carrier["carrier_code"] = carrier["Carrier_ID"].apply(clean)

carrier = carrier.dropna().drop_duplicates().reset_index(drop=True)
carrier.insert(0, "carrier_id", range(2000, 2000+len(carrier)))
carrier=carrier.drop(columns=['Carrier_ID'])
carrier


,carrier_id,carrier_code
0,2000,FOLW
1,2001,WWSP


In [ ]:

# =====================================
# 7. SHIP TO TABLE
# =====================================
ship = fills[["ShipTo", "ShipToName"]].copy()
ship["ship_to_id"] = pd.to_numeric(ship["ShipTo"], errors="coerce")
ship["ship_to_name"] = ship["ShipToName"].apply(clean)

ship = ship.dropna(subset=["ship_to_id"]).drop_duplicates("ship_to_id")
ship


,ShipTo,ShipToName,ship_to_id,ship_to_name
0,1500004425,"DOMINOS PIZZA LLC - EAST GRANDBY, C",1500004425,"DOMINOS PIZZA LLC - EAST GRANDBY, C"
4,1000015124,IT`LL BE PIZZA,1000015124,IT`LL BE PIZZA
6,2000008612,BIMBO HUNGRIA ALBANY,2000008612,BIMBO HUNGRIA ALBANY
8,2500000026,TREEHOUSE - FROZEN,2500000026,TREEHOUSE - FROZEN
9,20038136,CARAVAN,20038136,CARAVAN
...,...,...,...,...
435,2500068105,TRIPLE A BAGEL,2500068105,TRIPLE A BAGEL
441,2500001516,RESERS FINE FOODS,2500001516,RESERS FINE FOODS
452,20040348,ANGELS BAKERY,20040348,ANGELS BAKERY
473,1500046244,NARDI BAKERY INC,1500046244,NARDI BAKERY INC


In [ ]:

# =====================================
# 8. SALES ORDER (HEADER)
# =====================================
sales["order_no"] = sales["ORDER_NO"].apply(clean)

sales_order = sales[["order_no"]].drop_duplicates().reset_index(drop=True)
sales_order.insert(0, "order_id", range(3000, 3000+len(sales_order)))
sales_order


,order_id,order_no
0,3000,82378825
1,3001,M0060739
2,3002,M0291253
3,3003,M0289716
4,3004,M0291254
...,...,...
1320,4320,M0061332
1321,4321,M0061334
1322,4322,M0061529
1323,4323,M0061333


In [ ]:

# =====================================
# 9. SALES ORDER LINE
# =====================================
sales_line = sales[["ORDER_NO", "ITEM_NUM"]].copy()
sales_line.columns = ["order_no", "product_id"]

sales_line = sales_line.reset_index(drop=True)
sales_line.insert(0, "order_line_id", range(4000, 4000+len(sales_line)))
sales_line

,order_line_id,order_no,product_id
0,4000,82378825,5122073
1,4001,M0060739,06-000001
2,4002,M0291253,06-000001
3,4003,M0289716,06-000001
4,4004,M0291254,06-000001
...,...,...,...
1605,5605,M0061332,06-000001
1606,5606,M0061334,06-000001
1607,5607,M0061529,06-000001
1608,5608,M0061333,06-000001


In [ ]:

# =====================================
# 10. PACK RUN (FACT TABLE)
# =====================================
pack_run = pack.copy()

pack_run["product_id"] = pack_run["PRODUCT"].apply(clean)
pack_run["good_units"] = pd.to_numeric(pack_run["GoodUnits"], errors="coerce")

pack_run = pack_run.reset_index(drop=True)
pack_run.insert(0, "pack_run_id", range(5000, 5000+len(pack_run)))
pack_run


,pack_run_id,SITE_SHORT_NAME,PRODUCT,PRODUCT_DESC,LINE,PACKDATE,GoodUnits,TargetUnits,TotalUnits,CalcDT,MinutesRun,Pack OEE,product_id,good_units
0,5000,ALBANY-1001,5101204,APOLLO HG FLR 50LB-RK6,B&B Valve,2017-03-10,450.0,1024.0,454,17.937500,32.0,0.439453,5101204,450.0
1,5001,ALBANY-1001,5102879,BLUE STAR HG FLR 50LB-BI,B&B Valve,2017-03-10,1082.0,1760.0,1088,21.187500,55.0,0.614773,5102879,1082.0
2,5002,ALBANY-1001,5103937,CHIEF OF STAFF FLR 50LB-RK,B&B Valve,2017-03-10,911.0,2048.0,1023,35.531250,64.0,0.444824,5103937,911.0
3,5003,ALBANY-1001,5104949,CREMOSA FLR 50LB-RK7,B&B Valve,2017-03-10,723.0,2240.0,730,47.406250,70.0,0.322768,5104949,723.0
4,5004,ALBANY-1001,5108382,GOLDSTAR FLR 50LB-RK6,B&B Valve,2017-03-10,950.0,1376.0,952,13.312500,43.0,0.690407,5108382,950.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,5328,OGDEN-1025,5165273,HARVEST H&R FLR 50LB-RI (CH)-2029,Premier Tech,2017-03-22,330.0,350.0,358,2.000000,35.0,0.942857,5165273,330.0
329,5329,OGDEN-1025,NonProdDT,NaN,Premier Tech,2017-03-22,NaN,0.0,0,405.000000,405.0,0.000000,NonProdDT,NaN
330,5330,OGDEN-1025,5164147,(21314) ARDENT SPR WHITE WW FLR 50LB-AA-NWF,Premier Tech,2017-03-23,860.0,4410.0,908,253.571429,315.0,0.195011,5164147,860.0
331,5331,OGDEN-1025,5165251,HARVEST BREAD FLR 50LB-RI (CH)-405,Premier Tech,2017-03-23,650.0,1150.0,721,50.000000,115.0,0.565217,5165251,650.0


In [ ]:

# =====================================
# 11. MILL RUN
# =====================================
mill_run = mill.copy()

mill_run["production_mix"] = mill_run["ProductionMix"].apply(clean)
mill_run = mill_run.reset_index(drop=True)
mill_run.insert(0, "mill_run_id", range(6000, 6000+len(mill_run)))
mill_run


,mill_run_id,SITE_SHORT_NAME,MILLDATE,UNIT,ProductionMix,Calculated Downtime,MinRun,NODemand Downtime,Mill OEE,production_mix
0,6000,AYER-1004,2017-03-15,A,H54,4.971176,327.0,0.0,0.984798,H54
1,6001,AYER-1004,2017-03-15,A,H50,9.427941,1113.0,0.0,0.991529,H50
2,6002,AYER-1004,2017-03-15,B,M54,-5.563448,1440.0,0.0,1.003864,M54
3,6003,AYER-1004,2017-03-14,A,H50,1.900588,345.0,0.0,0.994491,H50
4,6004,AYER-1004,2017-03-14,B,M54,-28.433793,345.0,0.0,1.082417,M54
...,...,...,...,...,...,...,...,...,...,...
362,6362,OGDEN-1025,2017-03-21,A,AW42 CA46,4.430769,165.0,0.0,0.973147,AW42 CA46
363,6363,OGDEN-1025,2017-03-21,B,E45,11.274545,465.0,0.0,0.975754,E45
364,6364,OGDEN-1025,2017-03-21,B,D58,0.256364,75.0,0.0,0.996582,D58
365,6365,OGDEN-1025,2017-03-21,B,M56,-1.053636,10.0,0.0,1.105364,M56


In [ ]:

# =====================================
# 12. WORK ORDER
# =====================================
work["wo_no"] = work["WO_NO"].apply(clean)

work_order = work[["wo_no"]].copy()
work_order = work_order.dropna().drop_duplicates().reset_index(drop=True)
work_order.insert(0, "workorder_id", range(7000, 7000+len(work_order)))
work_order


,workorder_id,wo_no
0,7000,WO-100096409
1,7001,WO-100096410
2,7002,WO-100096421
3,7003,WO-100096036
4,7004,WO-100106740
...,...,...
623,7623,WO-100104624
624,7624,WO-100104625
625,7625,WO-100106652
626,7626,WO-100106656


In [ ]:

# =====================================
# 13. BIN TABLE
# =====================================
bin_df = bin_clean[["Bin_ID"]].copy()
bin_df["bin_id"] = bin_df["Bin_ID"].apply(clean)

bin_df = bin_df.dropna().drop_duplicates().reset_index(drop=True)
bin_df.insert(0, "bin_pk", range(8000, 8000+len(bin_df)))
bin_df


,bin_pk,Bin_ID,bin_id
0,8000,1AJK-22,1AJK-22
1,8001,1AJK-23,1AJK-23
2,8002,1AJK-24,1AJK-24
3,8003,1AJK-25,1AJK-25
4,8004,1AJK-26,1AJK-26
...,...,...,...
157,8157,1CHP-WW Bin 1,1CHP-WW Bin 1
158,8158,1CHP-WW Bin 2,1CHP-WW Bin 2
159,8159,1CHP-WW Bin 3,1CHP-WW Bin 3
160,8160,1CHP-WW Bin 4,1CHP-WW Bin 4


In [ ]:

# =====================================
# 14. FINAL OUTPUT (PRINT SAMPLE)
# =====================================
print("SITE\n", site.head())
print("PRODUCT\n", product.head())
print("CUSTOMER\n", customer.head())
print("CARRIER\n", carrier.head())
print("SALES ORDER\n", sales_order.head())
print("PACK RUN\n", pack_run.head())

SITE
    site_id site_name
0     1004      Ayer
1     1001    Albany
2     1025     Ogden
PRODUCT
    product_pk product_id                                       product_desc
0           1    5122073  ROSE HG FLR 50LB-RK7                          ...
1           2  06-000001  WHEAT MIDDLINGS LOOSE-BULK                    ...
2           3    5162628  WHITE SPRAY PASTRY FLR BULK-CB                ...
3           4    5113229  FOOD GRADE SOFT WHITE BRAN BULK-AA            ...
4           5    5127113  WHITE SNOW FLR 50LB-RA                        ...
CUSTOMER
    customer_id                                        CUSTOMER_NM  \
0         1000  ORLANDO FOODS INC                             ...   
1         1001  LARKIN CATTLE CO                              ...   
2         1002  TREEHOUSE PRIVATE BRANDS                      ...   
3         1003  THURSTON FOODS INC                            ...   
4         1004  S AND D MORREALE INC                          ...   

                cust